# Hafta 8: Yapay Sinir Ağlarının Temelleri

### Derin Öğrenme ile Görüntü İşleme — Yüksek Lisans Dersi

---

> *"What I cannot create, I do not understand."* — Richard Feynman  
> *"Yaratamadığımı anlamış sayılmam."*

---

## Bu Haftanın Yolculuğu

Sevgili öğrenciler, yedi hafta boyunca görüntülerle piksel düzeyinde konuştuk. Bu hafta, radikal bir soru soracağız:

> **"Ya makinenin kendisi bu filtreleri öğrenebilseydi?"**

Bu soru, bilgisayar bilimlerinin son yetmiş yılındaki en dönüştürücü fikirlerden birinin kapısını aralıyor. Bu hafta Frank Rosenblatt'ın 1958'deki "Perceptron" makinesinden başlayıp, çok katmanlı ağlara ve MNIST rakamlarını %97 doğrulukla tanıyan ilk sinir ağımıza kadar bir yolculuğa çıkacağız.

## Öğrenme Hedeflerimiz

Bu haftanın sonunda şunları yapabilir hâle geleceksiniz:

1. **Perceptron'un** matematiksel formülasyonunu yazıp geometrik anlamını açıklayabilmek
2. **XOR probleminin** neden tek katmanlı ağlarla çözülemediğini ispatlayabilmek  
3. **Çok Katmanlı Perceptron'un (MLP)** ileri yayılımını matris formunda hesaplayabilmek
4. **Aktivasyon fonksiyonlarının** rolünü ve neden gerekli olduklarını kavramak
5. **Kayıp fonksiyonu** ve **gradient descent** kavramlarını sezgisel olarak anlamak
6. **NumPy ile sıfırdan** bir sinir ağı yazıp MNIST üzerinde eğitebilmek

## Ön Gereksinimler

- **Lineer cebir:** matris çarpımı, transpoz, nokta çarpımı
- **Kısmi türev** (temel düzey)
- **Python, NumPy, matplotlib** (ilk 7 haftadan)

> **📚 Küçük bir not:** Lineer cebirinizi uzun süredir kullanmadıysanız, 3Blue1Brown'ın "Essence of Linear Algebra" serisinin ilk 4 videosu mükemmel bir tazeleyicidir. Bu hafta özellikle "matris çarpımı geometrik olarak doğrusal dönüşümdür" sezgisi işimize yarayacak.

## Bu Defterin Kullanımı

Bu defter üç ders boyunca (toplam 3 × 45 dk) işlenecek şekilde tasarlandı. Her ders kendi bölümüne ayrılmıştır:

- 📘 **Ders 1:** Perceptron ve Biyolojik Motivasyon
- 📗 **Ders 2:** Çok Katmanlı Perceptron ve Aktivasyon Fonksiyonları  
- 📙 **Ders 3:** Kayıp Fonksiyonları, Gradient Descent ve MNIST Laboratuvarı

Her bölümün sonunda bir özet, alıştırma ve ödev bulacaksınız. Kod hücrelerini **kendi elinizle çalıştırmayı, değiştirmeyi ve deneyler yapmayı** unutmayın. Bu defter izlemek için değil, *yapmak* için tasarlandı.


---

# 📘 Ders 1: Perceptron ve Biyolojik Motivasyon

*Süre: 45 dakika*

## 1.1 Başlangıç Sorusu: Neden Artık Filtre Tasarlamak İstemiyoruz?

İlk yedi haftada yaptığımız her şeyi düşünün. Sobel operatörünü hatırlayın:

$$G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}$$

Bu matrisi kim tasarladı? Irwin Sobel, 1968'de, Stanford'da. Peki neden tam olarak bu değerler? Çünkü Sobel, insan görsel sisteminin kenarlara nasıl tepki verdiğini gözlemledi, birkaç matematiksel ilke uyguladı ve bu değerleri **elle seçti**.

Şimdi size bir soru sorayım:

> **"Bir kediyi bir köpekten ayıran filtreyi elle tasarlayabilir misiniz?"**

Bir dakika düşünün. Kediler sivri kulaklı olur, köpekler daha yuvarlak. Ama sokak kedisinin kulağı yırtık olabilir. Bazı köpeklerin (Alman çobanı gibi) kulakları sivridir. Kedinin bıyıkları var, ama köpeğin de var. Renkler? İkisi de her renkte olabilir.

Sorun şu: **Dünyanın karmaşıklığını elle yazılmış kurallara sığdırmak imkânsız.** İşte sinir ağlarının devrimci fikri tam bu noktada doğdu: *Filtreyi elle yazmayalım, makine kendi öğrensin.*

---

## 1.2 Kısa Bir Tarih: Beyinden İlham Almak

### 🧠 1943: McCulloch ve Pitts — "Nöronun Mantığı"

Warren McCulloch bir nörofizyolog, Walter Pitts ise on altı yaşında matematik dehası bir sokak çocuğuydu (evet, gerçekten evsizdi ve üniversiteye bile gitmemişti, ama Chicago Üniversitesi'nde Bertrand Russell'ın *Principia Mathematica*'sını okuyup hatalarını bulmuştu). Birlikte çığır açan bir makale yayımladılar: *"A Logical Calculus of the Ideas Immanent in Nervous Activity."*

Bu ikilinin fikri şuydu: Beyindeki nöronlar, aslında **ikili (binary) mantık kapıları** gibi çalışıyor olabilir. Girişleri topla, bir eşiği aştıysa ateşle (1), aşmadıysa sus (0). Bu basit modelin AND, OR, NOT işlemlerini yapabildiğini, dolayısıyla *teorik olarak herhangi bir hesaplama yapabileceğini* gösterdiler.

> **💡 İlginç bilgi:** Walter Pitts'in hayatı trajik bir şekilde sona erdi. Arkadaşı Norbert Wiener ile yaşanan bir yanlış anlaşılma sonucu MIT'deki çevresinden koptu, alkolizme sürüklendi ve 1969'da 46 yaşında öldü. Ama McCulloch-Pitts nöronu, bugün kullandığımız her sinir ağının büyük büyükbabasıdır.

### 🤖 1958: Rosenblatt'ın Perceptron'u

Frank Rosenblatt, Cornell Üniversitesi'nde bir psikologdu. McCulloch-Pitts modelini bir adım öteye taşıdı: **öğrenebilen** bir nöron icat etti. Adını "Perceptron" koydu.

Rosenblatt Perceptron'u yalnızca kâğıt üzerinde bırakmadı; *gerçek bir makine* inşa etti. Mark I Perceptron adını verdiği bu cihaz, 400 fotoselli bir kameraya, analog potansiyometrelerle (ayarlanabilir dirençlerle) ağırlıkları saklayan bir devreye sahipti. Ağırlıklar, küçük elektrik motorlarıyla fiziksel olarak döndürülen potansiyometrelerdi!

The New York Times, 8 Temmuz 1958'de şu başlığı attı:

> *"NEW NAVY DEVICE LEARNS BY DOING: Psychologist Shows Embryo of Computer Designed to Read and Grow Wiser"*
>
> *(Yeni Deniz Kuvvetleri Cihazı Yaparak Öğreniyor: Psikolog, Okumak ve Bilgelik Kazanmak İçin Tasarlanmış Bilgisayarın Embriyosunu Gösterdi)*

Gazete ayrıca Rosenblatt'ın şu tahmininden bahsetti: Perceptron yakında "yürüyecek, konuşacak, görecek, yazacak, kendini çoğaltacak ve varlığının bilincine varacaktı."

> **⚠️ İbretlik not:** Yapay zekâ alanındaki *hype* (şişirme) döngüleri yeni bir fenomen değil. 1958'deki bu abartılı iddialar, sonraki yıllarda alanın başına gelecek "AI kış"larının habercisiydi. Bugün GPT-4 veya Stable Diffusion hakkında benzer iddialar duyduğunuzda, biraz durup düşünün — her hype döngüsü sonunda bir hesaplaşma getirir.


## 1.3 Perceptron'un Matematiği

Şimdi perceptron'un nasıl çalıştığına bakalım. Biyolojik nöronun şematiğini düşünün:

```
    Dendritler (girişler)          Hücre gövdesi            Akson (çıkış)
         x₁  ─────w₁────►
                               ┌─────────────┐
         x₂  ─────w₂────►      │  Σ + Eşik   │ ────────► y
                               └─────────────┘
         x₃  ─────w₃────►
```

Matematiksel olarak, bir perceptron şu işlemi yapar:

$$y = f\left(\sum_{i=1}^{n} w_i x_i + b\right)$$

### Formülün Ayrıntılı Açıklaması

Her bir sembolün ne anlama geldiğini tek tek inceleyelim:

- $x_i$: **i-inci girdi** (özellik). Örneğin bir görüntünün pikselleri ya da bir öğrencinin çalışma saati gibi.
- $w_i$: i-inci girdiye karşılık gelen **ağırlık** (weight). Bu değer, o girdinin ne kadar önemli olduğunu söyler. Büyük bir pozitif ağırlık "bu özellik çıktıyı artırır" demektir; büyük bir negatif ağırlık ise "bu özellik çıktıyı düşürür" der.
- $b$: **bias** (yanlılık, ön-eğilim). Bir nevi eşiği ayarlar. Türkçede "önyargı" diye çevrilse de matematiksel olarak daha çok "sabit terim" gibi düşünülmelidir.
- $\sum$: Tüm ağırlıklı girdileri toplar. Bu bir **doğrusal kombinasyondur**.
- $f(\cdot)$: **Aktivasyon fonksiyonu**. Klasik perceptron'da bu bir **basamak fonksiyonudur** (step function): toplam pozitifse 1, değilse 0 (veya -1) döndürür.
- $y$: **Çıktı**. Perceptron'un kararı.

### Vektör Formunda Yazımı

Aslında bu formülü çok daha derli toplu yazabiliriz. $\mathbf{w} = [w_1, w_2, ..., w_n]$ ve $\mathbf{x} = [x_1, x_2, ..., x_n]$ vektörleri ile:

$$y = f(\mathbf{w}^T \mathbf{x} + b)$$

Burada $\mathbf{w}^T \mathbf{x}$ **nokta çarpımı** (iç çarpım) demektir; iki vektörün eleman-eleman çarpılıp toplanması. Bu gösterim hem daha şık hem de kodlama açısından çok daha verimli (vektör operasyonları CPU'da tek seferde yapılır, `for` döngüsünden çok hızlıdır).

### Somut Bir Örnek: Ders Geçme Kararı

Soyutu somuta indirgeyelim. Diyelim bir öğrencinin dersi geçip geçmeyeceğini tahmin edecek bir perceptron kuralım:

- $x_1$: Haftalık çalışma saati
- $x_2$: Gecelik uyku saati  
- $x_3$: Önceki dersteki not ortalaması

Deneyimli bir profesör (siz!) bu faktörlere şu ağırlıkları verebilir:
- $w_1 = 0.6$ (çalışma çok önemli)
- $w_2 = 0.3$ (uyku önemli ama çalışma kadar değil)
- $w_3 = 0.5$ (önceki başarı iyi bir gösterge)
- $b = -2.0$ (geçmek için belli bir tabana ulaşılması lâzım)

Bir öğrenci haftada 5 saat çalışıyor, 7 saat uyuyor, önceki ortalaması 3.0 ise:

$$z = 0.6 \cdot 5 + 0.3 \cdot 7 + 0.5 \cdot 3.0 + (-2.0) = 3.0 + 2.1 + 1.5 - 2.0 = 4.6$$

$z > 0$ olduğu için $f(z) = 1$, yani "geçer" diyoruz.

> **🧩 Benzetme — Kabul Komitesi:** Perceptron'u bir üniversite kabul komitesi gibi düşünün. Komitedeki her üye (ağırlık) farklı bir kritere bakar: biri not ortalamasına, biri mülakat performansına, biri tavsiye mektuplarına. Her üye bir "kabul dozu" oyu verir. Bias, dekanın masanın üzerine koyduğu ön eğilimdir ("bu yıl daha seçiciyiz" veya "bu yıl kapıları açtık"). Tüm oylar toplanır, bir eşiği aşarsa aday kabul edilir. Perceptron'un öğrenmesi demek, komitenin zamanla hangi kriterlere ne kadar güvenmesi gerektiğini deneyimle öğrenmesi demektir.


In [ ]:
# Perceptron'u Python'da uygulayalım
import numpy as np
import matplotlib.pyplot as plt

# Görsel tutarlılık için stil ayarları
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

def perceptron(x, w, b):
    """
    Basit bir perceptron uygulamasi.
    
    Parametreler:
    -----------
    x : ndarray
        Giris vektoru, shape (n,)
    w : ndarray
        Agirlik vektoru, shape (n,)
    b : float
        Bias (skaler)
    
    Doner:
    ------
    int : 0 veya 1 (ikili siniflandirma)
    """
    z = np.dot(w, x) + b          # Agirlikli toplam + bias
    return 1 if z > 0 else 0      # Basamak fonksiyonu

# Ders gecme ornegini deneyelim
w = np.array([0.6, 0.3, 0.5])
b = -2.0

# Ogrenci 1: Calıskan
ogrenci1 = np.array([5, 7, 3.0])
print(f"Ogrenci 1 (5h calisma, 7h uyku, 3.0 ort): Karar = {perceptron(ogrenci1, w, b)}")

# Ogrenci 2: Az calısan
ogrenci2 = np.array([1, 5, 2.0])
print(f"Ogrenci 2 (1h calisma, 5h uyku, 2.0 ort): Karar = {perceptron(ogrenci2, w, b)}")

# Ogrenci 3: Sinirda
ogrenci3 = np.array([3, 6, 2.5])
print(f"Ogrenci 3 (3h calisma, 6h uyku, 2.5 ort): Karar = {perceptron(ogrenci3, w, b)}")

# Asagida z degerlerini de gorelim
print("\n--- Detayli analiz ---")
for i, ogr in enumerate([ogrenci1, ogrenci2, ogrenci3], 1):
    z = np.dot(w, ogr) + b
    karar = "GECER" if z > 0 else "KALIR"
    print(f"Ogrenci {i}: z = {z:+.2f}  -->  {karar}")


## 1.4 Geometrik Yorum: Perceptron Bir Çizgi Çeker

Buraya çok dikkat! Perceptron'u anlamanın en güçlü yolu, onu **geometrik** olarak görmektir.

İki boyutlu bir uzay düşünün ($x_1$ ve $x_2$ iki özellik olsun). Perceptron kararını şu denklemle verir:

$$w_1 x_1 + w_2 x_2 + b = 0$$

Bu denklemin ne olduğunu hatırlıyor musunuz? **Bir doğru denklemi!** 

Üç boyutta bu bir **düzlem** olur, $n$ boyutta ise **hiperdüzlem**. Bu hiperdüzlem, uzayı ikiye böler:

- Bir tarafında $z > 0$ (perceptron "1" der)
- Diğer tarafında $z < 0$ (perceptron "0" der)
- Üzerinde $z = 0$ (karar sınırı, *decision boundary*)

> **🎯 Kritik içgörü:** Perceptron, verileri ayıran bir doğru (veya hiperdüzlem) çizmeye çalışır. Ağırlıklar bu doğrunun **eğimini** (yönünü) belirler, bias ise **konumunu** (orijine uzaklığını) belirler.

### Ağırlık Vektörü Neyi Temsil Eder?

İşte çok güzel bir geometrik gerçek: $\mathbf{w}$ ağırlık vektörü, karar sınırına **dik** olan vektördür! Bu vektör, "pozitif tarafı" gösterir. Yani perceptron'un ağırlık vektörü, "1 kararı bu tarafta" diyen bir ok gibidir.

> **🧭 Benzetme — Deniz Feneri:** Ağırlık vektörünü bir deniz feneri ışığı gibi düşünün. Fener belli bir yönü aydınlatır; o yöndeki gemiler "güvenli" (sınıf 1), diğer taraftakiler "güvensiz" (sınıf 0) sayılır. Bias ise fenerin ne kadar uzağı aydınlattığını belirler — küçük bias varsa sadece çok yakın gemiler sınıf 1 olur, büyük (negatif) bias varsa çok daha uzağa kadar uzanır.

Aşağıda bunu görselleştirelim:


In [ ]:
# Perceptron'un karar sinirini gorsellestirelim
# AND ve OR mantik kapilarini yan yana gosterecegiz

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# =============================================================================
# AND Kapisi: (0,0)->0, (0,1)->0, (1,0)->0, (1,1)->1
# =============================================================================
w_and = np.array([1.0, 1.0])
b_and = -1.5  # Sadece her ikisi de 1 oldugunda toplam pozitif olur

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])

ax = axes[0]
for point, label in zip(X, y_and):
    color = '#E63946' if label == 1 else '#1D3557'
    marker = '^' if label == 1 else 'o'
    ax.scatter(point[0], point[1], c=color, marker=marker, s=300, 
               edgecolors='black', linewidth=2, zorder=3)

# Karar sinirini ciz: w1*x1 + w2*x2 + b = 0  =>  x2 = -(w1*x1 + b) / w2
x1_range = np.linspace(-0.5, 1.5, 100)
x2_boundary = -(w_and[0] * x1_range + b_and) / w_and[1]
ax.plot(x1_range, x2_boundary, 'g-', linewidth=2.5, label='Karar Siniri')

# Pozitif ve negatif bolgeleri golgelendir
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
zz = w_and[0] * xx + w_and[1] * yy + b_and
ax.contourf(xx, yy, zz, levels=[-10, 0, 10], colors=['#E8F4F8', '#FFF4E6'], alpha=0.6)

# Agirlik vektorunu ciz (karar sinirina dik)
mid_point = np.array([0.75, 0.75])
w_normalized = w_and / np.linalg.norm(w_and) * 0.3
ax.annotate('', xy=mid_point + w_normalized, xytext=mid_point,
            arrowprops=dict(arrowstyle='->', color='#8338EC', lw=2.5))
ax.text(1.05, 1.05, r'$\vec{w}$', fontsize=18, color='#8338EC', fontweight='bold')

ax.set_xlim(-0.3, 1.5)
ax.set_ylim(-0.3, 1.5)
ax.set_xlabel('$x_1$', fontsize=14)
ax.set_ylabel('$x_2$', fontsize=14)
ax.set_title('AND Kapisi\n(Dogrusal Olarak Ayrilabilir)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='upper right')
ax.set_aspect('equal')

# =============================================================================
# OR Kapisi: (0,0)->0, (0,1)->1, (1,0)->1, (1,1)->1
# =============================================================================
w_or = np.array([1.0, 1.0])
b_or = -0.5  # Herhangi biri 1 oldugunda toplam pozitif
y_or = np.array([0, 1, 1, 1])

ax = axes[1]
for point, label in zip(X, y_or):
    color = '#E63946' if label == 1 else '#1D3557'
    marker = '^' if label == 1 else 'o'
    ax.scatter(point[0], point[1], c=color, marker=marker, s=300,
               edgecolors='black', linewidth=2, zorder=3)

x2_boundary = -(w_or[0] * x1_range + b_or) / w_or[1]
ax.plot(x1_range, x2_boundary, 'g-', linewidth=2.5, label='Karar Siniri')

zz = w_or[0] * xx + w_or[1] * yy + b_or
ax.contourf(xx, yy, zz, levels=[-10, 0, 10], colors=['#E8F4F8', '#FFF4E6'], alpha=0.6)

ax.set_xlim(-0.3, 1.5)
ax.set_ylim(-0.3, 1.5)
ax.set_xlabel('$x_1$', fontsize=14)
ax.set_ylabel('$x_2$', fontsize=14)
ax.set_title('OR Kapisi\n(Dogrusal Olarak Ayrilabilir)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='upper right')
ax.set_aspect('equal')

plt.suptitle('Perceptron Karar Sinirlari: Dogrusal Ayrilabilir Problemler', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Mavi daire: sinif 0  |  Kirmizi ucgen: sinif 1")
print("Yesil cizgi: karar siniri (w1*x1 + w2*x2 + b = 0)")
print("Mor ok: agirlik vektoru (karar sinirina dik, pozitif tarafi gosterir)")


### 🧪 Hızlı Alıştırma

Şimdi kendiniz deneyin! Aşağıdaki hücrede NAND kapısı (AND'in tersi) için ağırlıkları ve bias'ı elle ayarlayın.

**NAND gerçek değer tablosu:**

| $x_1$ | $x_2$ | NAND |
|-------|-------|------|
| 0     | 0     | 1    |
| 0     | 1     | 1    |
| 1     | 0     | 1    |
| 1     | 1     | 0    |

> **💭 Düşünce ipucu:** NAND, AND'in tam tersi. AND için $w_1 = w_2 = 1, b = -1.5$ kullanmıştık. NAND için ağırlıkların işaretini ters çevirirsek ne olur?


In [ ]:
# NAND kapisi icin agirliklari bulun
# NAND: (0,0)->1, (0,1)->1, (1,0)->1, (1,1)->0

# Degerleri degistirip deneyin!
w_nand = np.array([-1.0, -1.0])  # AND'in ters isaretlisi
b_nand = 1.5                     # AND'in ters isaretlisi

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
beklenen = [1, 1, 1, 0]

print("NAND kapisi testi:")
print("-" * 45)
for i, x in enumerate(X):
    z = np.dot(w_nand, x) + b_nand
    y = 1 if z > 0 else 0
    dogru = "DOGRU" if y == beklenen[i] else "YANLIS"
    print(f"Giris: {x}  |  z = {z:+.2f}  |  Cikti: {y}  |  Beklenen: {beklenen[i]}  {dogru}")

# Bonus: Sadece 2 girisli tum mantik kapilari icin cozumleri toplayalim
print("\n" + "=" * 45)
print("Bazi mantik kapilari icin calisan agirliklar:")
print("=" * 45)
kapilar = {
    "AND":  (np.array([1.0, 1.0]),   -1.5),
    "OR":   (np.array([1.0, 1.0]),   -0.5),
    "NAND": (np.array([-1.0, -1.0]),  1.5),
    "NOR":  (np.array([-1.0, -1.0]),  0.5),
}
for ad, (w, b) in kapilar.items():
    print(f"{ad:5s}:  w = {w}, b = {b:+.1f}")


## 1.5 Perceptron Öğrenme Kuralı

Şimdiye kadar ağırlıkları **elle** belirledik. Rosenblatt'ın devrimci fikri şuydu: **Ağırlıklar otomatik olarak öğrenilebilir.** Algoritma inanılmaz derecede basittir:

### Perceptron Öğrenme Algoritması

Tüm eğitim verisi için, her örneği sırayla ele al:

1. Bir örnek $(\mathbf{x}, y_{\text{gerçek}})$ al
2. Tahmin yap: $y_{\text{tahmin}} = f(\mathbf{w}^T \mathbf{x} + b)$
3. Hata varsa ağırlıkları güncelle:

$$w_i \leftarrow w_i + \eta \cdot (y_{\text{gerçek}} - y_{\text{tahmin}}) \cdot x_i$$

$$b \leftarrow b + \eta \cdot (y_{\text{gerçek}} - y_{\text{tahmin}})$$

Burada $\eta$ (eta) **öğrenme oranıdır** (learning rate). Genellikle küçük bir pozitif sayıdır (0.01, 0.1 gibi).

### Bu Kuralın Sezgisel Anlamı

Güncelleme formülünü deşifre edelim. $(y_{\text{gerçek}} - y_{\text{tahmin}})$ üç değer alabilir:

- **0**: Tahmin doğru. Hiçbir şey değişmez. "Öğrenilecek bir şey yok."
- **+1**: Perceptron 0 demiş ama 1 olmalıydı. Bu durumda o girdinin ağırlığını **artırıyoruz** (eğer $x_i > 0$ ise). Yani "bu örneğe sıradaki sefer daha güçlü tepki ver" diyoruz.
- **-1**: Perceptron 1 demiş ama 0 olmalıydı. Ağırlıkları **azaltıyoruz**. "Bu örneğe sıradaki sefer daha az tepki ver."

> **⚖️ Benzetme — Deneyimli Hakim:** Perceptron'u bir hakim gibi düşünün. Her duruşmadan sonra "Bu kararım doğru muydu?" diye sorar. Yanlış bir mahkûmiyet verdiyse (0 olmalıyken 1 dedi), suçlamayı *azaltmayı* öğrenir. Masum birine hapis verdiyse (1 olmalıyken 0 dedi), delilleri *daha ciddiye almayı* öğrenir. Zamanla hakim daha iyi kararlar verir. İşte perceptron da böyledir — her hatasından bir şey öğrenir.

### Öğrenme Oranı ($\eta$) Neden Önemli?

Öğrenme oranı, "bir hatadan ne kadar ders çıkaracağımızı" belirler:

- **Çok büyük $\eta$**: Her hatada ağırlıkları sertçe değiştiririz. Sonuç: kararsız davranış, çözümün etrafında sekme veya ıraksama (*divergence*).
- **Çok küçük $\eta$**: Her hatada ağırlıklar azıcık değişir. Sonuç: çözüme ulaşmak çok yavaş olur.
- **İdeal $\eta$**: Yeterince hızlı öğrenen ama istikrarlı bir davranış.

> **🌡️ Benzetme — Termostat:** Öğrenme oranını bir termostat gibi düşünün. Odayı 22°C'de tutmak istiyorsunuz. Eğer termostatınız çok agresifse (yüksek $\eta$), 18°C'den 26°C'ye zıplar durur. Çok tembelse (düşük $\eta$), hedef sıcaklığa ulaşmak saatler alır. İyi bir termostat (iyi bir $\eta$), hızlı ama aşırıya kaçmadan yaklaşır.

### Rosenblatt'ın Yakınsama Teoremi (1962)

Rosenblatt matematiksel bir garanti de kanıtladı:

> **Eğer veri doğrusal olarak ayrılabiliyorsa, perceptron algoritması sonlu sayıda adımda bir çözüme yakınsar.**

Bu, duymak için harika bir cümle. Ama o koşul — "**eğer** doğrusal olarak ayrılabiliyorsa" — dikkatlerinizi çeksin. Bu koşul sağlanmazsa ne olur?

Bunun cevabı, yapay zekâ tarihinin en büyük trajedilerinden birine götürecek bizi.


In [ ]:
# Perceptron ogrenme algoritmasini sifirdan yazalim

class Perceptron:
    """
    Klasik Rosenblatt Perceptron'u.
    
    Tek bir yapay nöron - dogrusal ayrilabilir problemleri cozer.
    """
    
    def __init__(self, input_dim, learning_rate=0.1):
        """
        Parametreler:
        -----------
        input_dim : int
            Giris ozellik sayisi
        learning_rate : float
            Ogrenme orani (eta). Genellikle 0.01 - 0.5 arasi.
        """
        # Kucuk rastgele degerlerle basla
        self.w = np.random.randn(input_dim) * 0.1
        self.b = 0.0
        self.lr = learning_rate
        self.history = []  # Egitim surecini takip icin
    
    def predict(self, x):
        """Ileri yayilim: basamak fonksiyonu ile cikti."""
        z = np.dot(self.w, x) + self.b
        return 1 if z > 0 else 0
    
    def fit(self, X, y, epochs=100, verbose=False):
        """
        Perceptron ogrenme kurali ile egitim.
        
        Parametreler:
        -----------
        X : ndarray, shape (n_samples, input_dim)
            Egitim verisi
        y : ndarray, shape (n_samples,)
            Etiketler (0 veya 1)
        epochs : int
            Kac tur veri uzerinden gecilecek
        verbose : bool
            Ayrintili cikti goster
        """
        for epoch in range(epochs):
            hatalar = 0
            for xi, yi in zip(X, y):
                tahmin = self.predict(xi)
                hata = yi - tahmin
                
                if hata != 0:
                    # Rosenblatt'in ogrenme kurali
                    self.w += self.lr * hata * xi
                    self.b += self.lr * hata
                    hatalar += 1
            
            self.history.append(hatalar)
            
            if verbose and (epoch % 10 == 0 or hatalar == 0):
                print(f"  Epoch {epoch+1:3d}: {hatalar} hata")
            
            if hatalar == 0:
                print(f"Epoch {epoch+1}: Tum ornekler dogru siniflandirildi!")
                break
        else:
            print(f"{epochs} epoch sonunda hala {hatalar} hatali ornek var.")


# AND problemi uzerinde deneyelim
np.random.seed(42)  # Tekrarlanabilirlik icin

X_and = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)
y_and = np.array([0, 0, 0, 1])

print("=" * 50)
print("AND Kapisi Egitimi")
print("=" * 50)

p = Perceptron(input_dim=2, learning_rate=0.1)
p.fit(X_and, y_and, epochs=50, verbose=True)

print(f"\nOgrenilen agirliklar:")
print(f"  w = [{p.w[0]:+.3f}, {p.w[1]:+.3f}]")
print(f"  b = {p.b:+.3f}")

print(f"\nTahminler:")
for xi, yi in zip(X_and, y_and):
    pred = p.predict(xi)
    durum = "DOGRU" if pred == yi else "YANLIS"
    print(f"  x = {xi}  |  tahmin = {pred}  |  gercek = {yi}  {durum}")


## 1.6 XOR Felaketi: Bir Devri Kapatan Problem

1969'da iki MIT profesörü, **Marvin Minsky** ve **Seymour Papert**, *"Perceptrons: An Introduction to Computational Geometry"* adlı kitaplarını yayımladılar. Bu kitap, perceptron'un sınırlarını matematiksel olarak ortaya koyuyordu. En çarpıcı örnek: **XOR problemi**.

XOR (exclusive OR, "özel VEYA") şu tabloya sahiptir:

| $x_1$ | $x_2$ | XOR |
|-------|-------|-----|
| 0     | 0     | 0   |
| 0     | 1     | 1   |
| 1     | 0     | 1   |
| 1     | 1     | 0   |

"İkisinden tam biri 1 ise 1, değilse 0." Mantıksal olarak çok basit görünür. Ama geometrik olarak bakalım:


In [ ]:
# XOR'u gorsellestirelim - neden dogrusal olarak ayrilamaz?
fig, ax = plt.subplots(figsize=(8, 8))

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

for point, label in zip(X, y_xor):
    color = '#E63946' if label == 1 else '#1D3557'
    marker = '^' if label == 1 else 'o'
    ax.scatter(point[0], point[1], c=color, marker=marker, s=500,
               edgecolors='black', linewidth=2.5, zorder=3)
    ax.annotate(f'({point[0]},{point[1]}) $\\rightarrow$ {label}', 
                xy=point, xytext=(point[0]+0.06, point[1]+0.06),
                fontsize=13, fontweight='bold')

# Birkac olası "cozmeye calisan" dogru cizelim
x1_range = np.linspace(-0.3, 1.3, 100)

ax.plot(x1_range, 0.5 * np.ones_like(x1_range), 'g--', linewidth=2, 
        label='Deneme 1: yatay cizgi', alpha=0.7)
ax.plot(0.5 * np.ones_like(x1_range), x1_range, 'm--', linewidth=2,
        label='Deneme 2: dikey cizgi', alpha=0.7)
ax.plot(x1_range, x1_range, color='orange', linestyle='--', linewidth=2,
        label='Deneme 3: diyagonal', alpha=0.7)
ax.plot(x1_range, 1 - x1_range, color='purple', linestyle='--', linewidth=2,
        label='Deneme 4: ters diyagonal', alpha=0.7)

ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('$x_1$', fontsize=15)
ax.set_ylabel('$x_2$', fontsize=15)
ax.set_title('XOR Problemi: Hicbir Tek Dogru Bu Noktalari Ayiramaz!', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='lower right')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print("\nKirmizi ucgen: sinif 1  |  Mavi daire: sinif 0")
print("\nDikkat edin:")
print("  Kirmizi noktalar bir kosegen uzerinde: (0,1) ve (1,0)")
print("  Mavi noktalar diger kosegen uzerinde:  (0,0) ve (1,1)")
print("\nHicbir DUZ CIZGI bu iki kumeyi ayiramaz!")
print("Bu matematiksel olarak kanitlanabilir (Minsky & Papert 1969).")


In [ ]:
# Perceptron'u XOR uzerinde egitmeyi deneyelim - basarisiz olmali

print("=" * 50)
print("XOR Problemi ile Mucadele")
print("=" * 50)

np.random.seed(42)
X_xor = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

p_xor = Perceptron(input_dim=2, learning_rate=0.1)
p_xor.fit(X_xor, y_xor, epochs=200)

print(f"\nOgrenilen agirliklar: w = {p_xor.w}, b = {p_xor.b:.3f}")
print("\nTahminler:")
for xi, yi in zip(X_xor, y_xor):
    pred = p_xor.predict(xi)
    durum = "DOGRU" if pred == yi else "YANLIS"
    print(f"  x = {xi}  |  tahmin = {pred}  |  gercek = {yi}  {durum}")

# Hata grafigi
plt.figure(figsize=(11, 4))
plt.plot(p_xor.history, linewidth=2, color='crimson')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Yanlis Siniflandirilan Ornek Sayisi', fontsize=12)
plt.title('XOR uzerinde Perceptron: Hata Asla Sifira Inmez!', 
          fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='green', linestyle='--', alpha=0.5, label='Ideal (hata=0)')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("\nSONUC: Perceptron sonsuza kadar hata yapmaya devam eder.")
print("Bu bir bug degil - matematiksel bir imkansizlik!")


## 1.7 AI Kışı: Bir Alanın 15 Yıl Uykuya Dalışı

Minsky ve Papert'ın kitabı sadece bir teorik eleştiri değildi; bir nevi ölüm ilanıydı. Kitapta şu satırlar vardı:

> *"Perceptron'un geleceği hakkında iyimserlik için hiçbir dayanak göremiyoruz."*

Bu satırlar bilim camiasında dalga dalga yayıldı. ABD savunma araştırma ajansı DARPA, sinir ağı araştırmalarına verdiği fonları büyük ölçüde kesti. 1970'ler boyunca sinir ağı araştırmaları "mahalle çocukları" tarafından yapılan marjinal bir aktiviteye dönüştü. Bu döneme literatürde **"First AI Winter"** (Birinci Yapay Zekâ Kışı) denir.

> **🕰️ Tarihsel ironi:** Minsky ve Papert aslında çok katmanlı ağların XOR'u çözebileceğini *biliyorlardı*. Hatta kitapta bunu belirtiyorlardı. Ama çok katmanlı ağların nasıl **eğitileceği** (geri yayılım) o zamanki araştırmacılar tarafından bilinmiyordu ve Minsky'ye göre "çözümsüz" görünüyordu. Tarih onu haksız çıkaracaktı.

> **💔 Hüzünlü bir not:** Frank Rosenblatt bu kışın sonunu göremedi. 1971'de, 43. doğum gününde Chesapeake Körfezi'nde bir tekne kazasında hayatını kaybetti. Ne 1986'daki geri yayılım devrimini, ne 2012'deki derin öğrenme patlamasını gördü. Bugün ImageNet'i %99 doğrulukla sınıflandıran her sinir ağı, aslında onun 1958'de gördüğü hayalin gerçekleşmiş hâlidir.

1986'ya kadar beklememiz gerekecek. O yıl, David Rumelhart, Geoffrey Hinton ve Ronald Williams *Nature* dergisinde bir makale yayımladılar: *"Learning representations by back-propagating errors."* Bu makale, çok katmanlı ağları eğitmenin matematiksel reçetesini veriyordu. Alan yeniden uyandı.

### XOR'u Nasıl Çözeriz? İlk İpucu

Peki XOR gerçekten çözülemez mi? Hayır — sadece **tek bir perceptron** ile çözülemez. Cevap çok basit:

> **Eğer bir doğru yetmiyorsa, iki doğru kullan!**

Düşünün: XOR'daki kırmızı noktaları bir *ikiz doğrular* sandviçi olarak ayırabiliriz. Bir doğru "$x_1 + x_2 \geq 0.5$" der (yani en az biri 1), diğeri "$x_1 + x_2 \leq 1.5$" der (yani ikisi birden 1 değil). Bu iki koşulun **kesişimi** tam olarak XOR'un 1 dediği noktaları verir!

Bunu şöyle de yazabiliriz:

$$\text{XOR}(x_1, x_2) = \text{AND}(\text{OR}(x_1, x_2), \text{NAND}(x_1, x_2))$$

İşte fikir bu: **Bir perceptron katmanının çıktısını, başka bir perceptron katmanına besleyelim.** Böylece ikinci katman, birinci katmanın "dediklerini" birleştirerek daha karmaşık kararlar verebilir. 

Bu fikir, bizi **Çok Katmanlı Perceptron**'a götürür — ve bir sonraki dersimizin konusu tam olarak budur.

---

### 📌 Ders 1 Özeti

| Kavram | Özet |
|---|---|
| **Perceptron** | Biyolojik nörondan esinlenen ilk öğrenebilen makine (Rosenblatt, 1958) |
| **Matematik** | $y = f(\mathbf{w}^T \mathbf{x} + b)$ — ağırlıklı toplam + aktivasyon |
| **Geometri** | Uzayı ikiye bölen bir **hiperdüzlem** çizer |
| **Öğrenme kuralı** | $w_i \leftarrow w_i + \eta (y_{\text{gerçek}} - y_{\text{tahmin}}) x_i$ |
| **Yakınsama** | Veri doğrusal ayrılabiliyorsa sonlu adımda çözümü bulur |
| **Sınırı** | **Doğrusal olmayan** problemleri (XOR gibi) çözemez |
| **Tarih** | Bu sınırlama, AI'ı 1969-1986 arası uykuya yatırdı |

### 📝 Ödev (Ders 1 için)

1. **Programlama:** Kendi `Perceptron` sınıfınızı yazın (yukarıdaki koda bakmadan, kendi başınıza). Yazdıktan sonra karşılaştırın.
2. **Deneyler:** AND, OR, NAND, NOR kapıları için ayrı ayrı eğitin ve öğrenilen ağırlıkları raporlayın. Her biri kaç epoch'ta yakınsadı?
3. **Başarısızlık raporu:** XOR için eğitin, 500 epoch çalıştırın, hatanın nasıl davrandığını grafik ile gösterin.
4. **🧠 Düşünce deneyi:** XOR'u iki perceptron'un çıktısını üçüncü bir perceptron'a besleyerek çözebilir misiniz? Kâğıt-kalem ile ağırlıkları bulun. (İpucu: Birinci katmanın biri OR hesaplasın, diğeri NAND. İkinci katman bu ikisinin AND'ini alsın.)
5. **🔍 Araştırma:** "AI Winter" terimini araştırın. Kaç tane AI kışı yaşandı? Hangisi bu perceptron probleminden tetiklendi?

---


# 📗 Ders 2: Çok Katmanlı Perceptron ve Aktivasyon Fonksiyonları

*Süre: 45 dakika*

## 2.1 XOR'un Çözümü: İki Doğrudan Biri

Önceki derste XOR'un tek perceptronla çözülemediğini gördük. Şimdi bu tuhaf probleme başka bir gözle bakalım. 

Perceptron'u şöyle düşünmüştük: uzayda bir **doğru** çizer. Peki ya **iki doğru** çizseydik? Alttaki görselde XOR problemini iki doğruyla ayırmayı deneyelim:


In [ ]:
# XOR problemini IKI dogru ile ayirmaya calisalim
fig, ax = plt.subplots(figsize=(9, 9))

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

# Iki dogru:
# Dogru 1 (OR gibi): x1 + x2 - 0.5 = 0   -> bu dogrunun USTUNDEKI 3 nokta "en az biri 1"
# Dogru 2 (NAND gibi): x1 + x2 - 1.5 = 0 -> bu dogrunun ALTINDAKI 3 nokta "ikisi birden degil"
# Bu ikisinin KESISIMI: tam olarak XOR'un 1 dedikleri

x1_range = np.linspace(-0.3, 1.3, 100)
# x1 + x2 = 0.5 => x2 = 0.5 - x1
line1 = 0.5 - x1_range
# x1 + x2 = 1.5 => x2 = 1.5 - x1  
line2 = 1.5 - x1_range

ax.plot(x1_range, line1, 'g-', linewidth=2.5, label=r'Dogru 1: $x_1 + x_2 = 0.5$ (OR sınırı)')
ax.plot(x1_range, line2, 'b-', linewidth=2.5, label=r'Dogru 2: $x_1 + x_2 = 1.5$ (NAND sınırı)')

# Iki dogru arasindaki bolgeyi golgelendir (bu bolge sinif 1)
xx, yy = np.meshgrid(np.linspace(-0.3, 1.3, 300), np.linspace(-0.3, 1.3, 300))
# Sinif 1 bolgesi: 0.5 < x1+x2 < 1.5
mask = ((xx + yy) > 0.5) & ((xx + yy) < 1.5)
ax.contourf(xx, yy, mask.astype(float), levels=[0.5, 1.5], 
            colors=['#FFE8E8'], alpha=0.6)

# Noktalari ciz
for point, label in zip(X, y_xor):
    color = '#E63946' if label == 1 else '#1D3557'
    marker = '^' if label == 1 else 'o'
    ax.scatter(point[0], point[1], c=color, marker=marker, s=500,
               edgecolors='black', linewidth=2.5, zorder=3)
    ax.annotate(f'({point[0]},{point[1]}) $\\rightarrow$ {label}',
                xy=point, xytext=(point[0]+0.06, point[1]+0.06),
                fontsize=13, fontweight='bold')

ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('$x_1$', fontsize=15)
ax.set_ylabel('$x_2$', fontsize=15)
ax.set_title('XOR = (OR) VE (NAND)\nIki dogrunun KESISIMI sinif 1 bolgesini verir',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print("\nGozlem: Pembe bolge (iki dogru arasi) kirmizi noktalari icerir.")
print("Bu bolge tam olarak 'en az biri 1 AMA ikisi birden 1 degil' demektir.")
print("Bu da XOR'un tanimidir!")


## 2.2 Çok Katmanlı Perceptron (MLP) Doğuyor

Yukarıdaki görsel bize şunu söylüyor: **İki perceptron'un çıktısını bir üçüncü perceptron'a beslersek, XOR'u çözebiliriz.**

Mimariye bakalım:

```
  Giris Katmani       Gizli Katman        Cikis Katmani
  (2 noron)           (2 noron)           (1 noron)

       x1 ──┬─────► [ P1: OR  ] ──h1──┐
            │                          │
            │                          ├──► [ P3: AND ] ──► y
            │                          │
       x2 ──┴─────► [ P2: NAND ] ─h2──┘
```

Bu yapıya **Multi-Layer Perceptron (MLP)** — yani **Çok Katmanlı Perceptron** denir. Her nöronun çıktısı, bir sonraki katmandaki nöronların girdisi olur.

### Neden "Gizli" Katman?

Orta katmana "gizli" (hidden) denir, çünkü dışarıdan doğrudan görünmez. Sadece **girdiyi** ve **çıktıyı** dışarıdan görürüz; aradakiler ağın iç hesap mekanizmasıdır.

> **🎭 Benzetme — Jüri ve Alt Jüriler:** MLP'yi bir yarışma düşünün. Bir film festivalinde, "En İyi Film" ödülünü tek bir jüri vermek yerine şöyle yapsalardı: iki alt jüri (senaryo jürisi ve görüntü jürisi) filmi değerlendirir, her biri bir puan verir. Sonra bir üst jüri, bu iki puanı bakarak nihai kararı verir. Gizli katman, "alt jüriler"; çıktı katmanı, "üst jüri"dir. Gizli katman ham veriden daha "anlamlı" özellikler çıkarır; çıktı katmanı bu özellikleri birleştirerek karar verir.

### Manuel Ağırlıklarla XOR Çözümü

Şimdi bu mimariyi manuel ağırlıklarla inşa edelim. Her perceptron'u öğrendiğimiz kurallarla yazacağız:

- **P1 (OR):** $w = [1, 1], b = -0.5$
- **P2 (NAND):** $w = [-1, -1], b = 1.5$  
- **P3 (AND):** $w = [1, 1], b = -1.5$ (ama giriş olarak $h_1, h_2$'yi alacak)

Kodlayalım ve çalıştıralım:


In [ ]:
# Manuel agirliklarla XOR'u cozen bir MLP insa edelim

def step(z):
    """Basamak fonksiyonu: pozitifse 1, degilse 0."""
    return 1 if z > 0 else 0

def xor_mlp(x1, x2):
    """
    El ile tasarlanmis bir MLP ile XOR hesaplar.
    
    Mimari:
      Giris (2) -> Gizli (2) -> Cikis (1)
    """
    # === Gizli katman ===
    # Noron 1: OR
    h1 = step(1.0 * x1 + 1.0 * x2 - 0.5)
    # Noron 2: NAND
    h2 = step(-1.0 * x1 - 1.0 * x2 + 1.5)
    
    # === Cikis katmani ===
    # Noron 3: AND (h1 ve h2 uzerinde)
    y = step(1.0 * h1 + 1.0 * h2 - 1.5)
    
    return y, h1, h2

# Test edelim
print("XOR'u MLP ile cozme:")
print("=" * 55)
print(f"{'x1':>3} {'x2':>3} | {'h1 (OR)':>8} {'h2 (NAND)':>10} | {'y (XOR)':>8} | beklenen")
print("-" * 55)

beklenen_xor = {(0,0): 0, (0,1): 1, (1,0): 1, (1,1): 0}
for x1 in [0, 1]:
    for x2 in [0, 1]:
        y, h1, h2 = xor_mlp(x1, x2)
        exp = beklenen_xor[(x1, x2)]
        durum = "OK" if y == exp else "HATA"
        print(f"{x1:>3} {x2:>3} | {h1:>8} {h2:>10} | {y:>8} | {exp}  {durum}")

print("\n" + "=" * 55)
print("BASARI! Iki katmanli ag XOR'u cozdu.")
print("Tek katmanin beceremedigini iki katman basardi.")


## 2.3 Evrensel Yaklaşım Teoremi: MLP'lerin Süper Gücü

Bir matematikçi, *Cybenko* (1989) ve daha sonra *Hornik* (1991), olağanüstü bir şey kanıtladı:

> **Universal Approximation Theorem (Evrensel Yaklaşım Teoremi):**  
> Tek gizli katmanlı ama **yeterli sayıda nöron içeren** bir MLP, herhangi bir sürekli fonksiyonu istenen doğrulukta yaklaştırabilir.

Bu cümle çok güçlü. "Herhangi bir sürekli fonksiyon" deyince akla gelen her şey: resim tanıma, ses tanıma, dil çevirisi, fiyat tahmini... Tüm bunlar aslında bir giriş-çıkış fonksiyonudur. Ve teorik olarak, yeterince büyük bir MLP bunları öğrenebilir.

### Ama Dikkat: Bir Tuzak Var

Teorem "yeterli sayıda nöron" diyor ama kaç tane olduğunu söylemiyor. Pratikte bu sayı, astronomik olabilir. Ayrıca teorem **bir ağın var olduğunu** söylüyor, ama onu *nasıl eğiteceğimizi* söylemiyor.

> **🏗️ Benzetme — Evren ve Tuğlalar:** "Yeterli sayıda tuğlayla her binayı inşa edebilirsiniz" demek matematiksel olarak doğrudur. Ama pratikte:
> 1. Piramit inşa etmek için 2 milyon tuğla gerekir (inşa etmek zor).
> 2. Hangi tuğlayı nereye koyacağınızı bilmek başka bir problemdir.
>
> Derin öğrenmenin "derin" olması (tek geniş katman yerine çok katman) işte bu iki pratik sorunu çözer: **derin ağlar aynı işi çok daha az nöronla yapabilir**, ve **katmanlı öğrenme daha kolay optimize edilir**.

Bu yüzden modern derin öğrenmede 1 değil, **onlarca, yüzlerce, hatta binlerce** katman kullanıyoruz. ResNet-152 tam 152 katmanlıdır, GPT-3 ise 96 transformer katmanı içerir.

---

## 2.4 Aktivasyon Fonksiyonları: Neden Gerekli?

Şimdi kritik bir soruya geliyoruz. Hatırlayın, perceptron'un tanımında $f(\cdot)$ aktivasyon fonksiyonu vardı. Neden bunu kullanıyoruz? Hiç kullanmasak ne olur?

Diyelim aktivasyon yok, sadece doğrusal işlem yapıyoruz. İki katmanlı bir ağ:

$$\mathbf{h} = \mathbf{W}^{(1)} \mathbf{x} + \mathbf{b}^{(1)}$$
$$\mathbf{y} = \mathbf{W}^{(2)} \mathbf{h} + \mathbf{b}^{(2)}$$

Bunları birleştirelim:

$$\mathbf{y} = \mathbf{W}^{(2)} (\mathbf{W}^{(1)} \mathbf{x} + \mathbf{b}^{(1)}) + \mathbf{b}^{(2)} = \underbrace{\mathbf{W}^{(2)} \mathbf{W}^{(1)}}_{\mathbf{W}'} \mathbf{x} + \underbrace{\mathbf{W}^{(2)} \mathbf{b}^{(1)} + \mathbf{b}^{(2)}}_{\mathbf{b}'} = \mathbf{W}' \mathbf{x} + \mathbf{b}'$$

### 💥 Büyük İçgörü

İki doğrusal katmanın bileşkesi, **tek bir doğrusal katmandır!** Yani aktivasyon fonksiyonu olmadan, ne kadar çok katman eklersek ekleyelim, elimizde yine sadece bir perceptron olur.

> **🔗 Benzetme — Çevirmen Zinciri:** Türkçeden İngilizceye çeviren bir çevirmen, sonra İngilizceden Fransızcaya çeviren başka bir çevirmen. Eğer her ikisi de "çeviriyi aynen geçir" yaparsa (doğrusal davranır), sonuç olarak elde ettiğiniz şey **Türkçeden Fransızcaya çeviri yapan tek bir çevirmen**dir. Zincir bir çevirmenin işini görür. Aktivasyon fonksiyonu, ara çevirmenin kendi yorumunu, vurgularını katması gibidir — ancak bu sayede zincirin bütünü, tek bir çevirmenin yapamayacağı şeyler yapar.

**Sonuç:** Derin öğrenmenin derinliği, doğrusal olmayan aktivasyon fonksiyonları sayesinde anlam kazanır.


## 2.5 Aktivasyon Fonksiyonlarının Zoolojisi

Şimdi en sık kullanılan aktivasyon fonksiyonlarını tanıyalım. Her birinin kendine özgü kişiliği vardır.

### 🔵 Sigmoid (Lojistik Fonksiyon)

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

- **Çıktı aralığı:** $(0, 1)$
- **Türevi:** $\sigma'(x) = \sigma(x)(1 - \sigma(x))$

**Formülün açıklaması:** $e^{-x}$ ifadesi negatif $x$'ler için çok büyür (örn. $x=-5$ iken $e^5 \approx 148$), pozitif $x$'ler için sıfıra yaklaşır (örn. $x=5$ iken $e^{-5} \approx 0.007$). Dolayısıyla payda negatif $x$'ler için büyük, pozitif $x$'ler için $\approx 1$. Sonuç: çıktı negatiflerde 0'a, pozitiflerde 1'e yaklaşır, ortada (x=0'da) tam 0.5.

**Avantajları:**
- Olasılık yorumu yapılabilir (0 ile 1 arası)
- Her yerde türevlenebilir

**Dezavantajları:**
- **Vanishing gradient:** Uçlarda türev sıfıra yaklaşır (0'a çok yakın değerler). Derin ağlarda geri yayılım yaparken bu sıfırlar çarpıldıkça gradyan "kaybolur" ve ağın baş katmanları öğrenmez hale gelir.
- Sıfır-merkezli değildir (hep pozitif çıktı verir), bu da optimizasyonu zorlaştırır.

### 🟠 Tanh (Hiperbolik Tanjant)

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} = 2\sigma(2x) - 1$$

- **Çıktı aralığı:** $(-1, 1)$
- **Türevi:** $\tanh'(x) = 1 - \tanh^2(x)$

Sigmoid'in "ölçeklenip kaydırılmış" hâli gibidir. Sıfır-merkezlidir (bu, optimizasyonu kolaylaştırır), ama **vanishing gradient** sorunu sigmoid gibi onda da vardır.

### 🔴 ReLU (Rectified Linear Unit)

$$\text{ReLU}(x) = \max(0, x)$$

2012'de AlexNet ile popülerleşti ve derin öğrenme devrimini başlatan aktivasyondur. Aşırı basit görünmesine rağmen olağanüstü etkilidir.

- **Çıktı aralığı:** $[0, \infty)$
- **Türevi:** $x > 0$ için 1, $x < 0$ için 0 (x=0'da tanımsız, pratikte 0 alınır)

**Avantajları:**
- Hesaplama ucuz (bir karşılaştırma!)
- Pozitif tarafta vanishing gradient yok (türev sabit 1)
- Seyrek (sparse) aktivasyonlar üretir (çoğu çıktı tam 0)

**Dezavantajı:**
- **Dying ReLU:** Bir nöronun girdisi sürekli negatif olursa, türev sürekli 0 olur, nöron "ölür" ve bir daha öğrenmez.

> **🧟 Benzetme — Uyanmayan Nöron:** ReLU nöronunu tembel bir öğrenci gibi düşünün. Eğer motivasyonu (girdisi) hep negatifse, hiç çalışmaz (çıktı 0) ve geri yayılımda uyandırma sinyali de alamaz. Zamanla tamamen pasifleşir. "Dying ReLU" böyledir.

### 🟢 Leaky ReLU

$$\text{LeakyReLU}(x) = \begin{cases} x & x > 0 \\ \alpha x & x \leq 0 \end{cases}$$

Genellikle $\alpha = 0.01$ seçilir. Negatif tarafa küçük bir eğim vererek "dying ReLU" problemini çözer.

### 🔷 GELU (Gaussian Error Linear Unit)

$$\text{GELU}(x) = x \cdot \Phi(x)$$

Burada $\Phi(x)$ standart normal dağılımın kümülatif dağılım fonksiyonudur. Yumuşak bir ReLU versiyonudur ve modern Transformer modellerinde (BERT, GPT) standart hâline gelmiştir.

Görsel olarak hepsine bakalım:


In [ ]:
# Aktivasyon fonksiyonlarini ve turevlerini gorsellestirelim

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1 - np.tanh(x)**2

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def leaky_relu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)

def leaky_relu_derivative(x, alpha=0.01):
    return np.where(x > 0, 1, alpha)

def gelu(x):
    # Yaklasik form - daha hizli
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

# Gorsellestir
x = np.linspace(-5, 5, 500)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Fonksiyonlar
ax = axes[0, 0]
ax.plot(x, sigmoid(x), label='Sigmoid', linewidth=2.5, color='#3A86FF')
ax.plot(x, tanh(x), label='Tanh', linewidth=2.5, color='#FB5607')
ax.plot(x, relu(x), label='ReLU', linewidth=2.5, color='#E63946')
ax.plot(x, leaky_relu(x, 0.1), label='Leaky ReLU (a=0.1)', linewidth=2.5, color='#06A77D', linestyle='--')
ax.plot(x, gelu(x), label='GELU', linewidth=2.5, color='#8338EC')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_title('Aktivasyon Fonksiyonlari', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_ylim(-1.5, 5)

# 2. Turevler
ax = axes[0, 1]
ax.plot(x, sigmoid_derivative(x), label="Sigmoid'", linewidth=2.5, color='#3A86FF')
ax.plot(x, tanh_derivative(x), label="Tanh'", linewidth=2.5, color='#FB5607')
ax.plot(x, relu_derivative(x), label="ReLU'", linewidth=2.5, color='#E63946')
ax.plot(x, leaky_relu_derivative(x, 0.1), label="Leaky ReLU'", 
        linewidth=2.5, color='#06A77D', linestyle='--')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel("f'(x)", fontsize=12)
ax.set_title('Aktivasyon Fonksiyonlarinin Turevleri', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)

# 3. Sigmoid odakli yakinlastirma
ax = axes[1, 0]
ax.plot(x, sigmoid(x), label='Sigmoid', linewidth=2.5, color='#3A86FF')
ax.plot(x, sigmoid_derivative(x), label="Sigmoid' (turev)", 
        linewidth=2.5, color='#3A86FF', linestyle='--')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
# Vanishing gradient bolgelerini isaretle
ax.axvspan(-5, -3, alpha=0.15, color='red', label='Vanishing gradient bolgesi')
ax.axvspan(3, 5, alpha=0.15, color='red')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('deger', fontsize=12)
ax.set_title('Sigmoid: Vanishing Gradient Problemi', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 4. ReLU ve Leaky ReLU
ax = axes[1, 1]
ax.plot(x, relu(x), label='ReLU', linewidth=2.5, color='#E63946')
ax.plot(x, leaky_relu(x, 0.1), label='Leaky ReLU (a=0.1)', 
        linewidth=2.5, color='#06A77D')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.axvspan(-5, 0, alpha=0.15, color='red', label='ReLU burada "olur"')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_title('ReLU vs Leaky ReLU: Dying Neuron Problemi', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-1, 5)

plt.suptitle('Aktivasyon Fonksiyonlari: Derin Ogrenmenin Dogrusal Olmayanlari',
             fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\nOzet:")
print("  Sigmoid, Tanh: Uclarda turev ~ 0 (vanishing gradient)")
print("  ReLU:          Negatif bolgede 'olu' olabilir (dying ReLU)")
print("  Leaky ReLU:    Negatif bolgede kucuk egim verir, olmez")
print("  GELU:          ReLU'nun yumusak versiyonu, Transformer'larda standart")


## 2.6 İleri Yayılımın Matris Formülasyonu

Şimdiye kadar tek bir nöron üzerinde çalıştık. Ama gerçek bir ağda onlarca, yüzlerce nöron var ve her katmanda aynı işlem tekrar ediyor. `for` döngüleriyle yazmak hem yazım açısından dağınık olurdu hem de çok yavaş çalışırdı.

İşte **lineer cebir** imdadımıza yetişiyor. Tüm katmanı tek bir matris işlemiyle ifade edebiliriz.

### Tek Katman Formülü

Bir katmanda şu işlem gerçekleşir:

$$\boxed{\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}}$$

$$\boxed{\mathbf{a}^{[l]} = f(\mathbf{z}^{[l]})}$$

### Sembollerin Tam Açıklaması

- $[l]$: Katman indeksi. $[0]$ giriş katmanı, $[L]$ çıkış katmanı.
- $\mathbf{a}^{[l-1]}$: Önceki katmanın **aktivasyonu** (çıktısı). Boyut: $(n^{[l-1]} \times 1)$ — yani önceki katmandaki nöron sayısı uzunluğunda bir sütun vektör.
- $\mathbf{W}^{[l]}$: Bu katmanın **ağırlık matrisi**. Boyut: $(n^{[l]} \times n^{[l-1]})$ — yani satır sayısı = bu katmandaki nöron sayısı, sütun sayısı = önceki katmandaki nöron sayısı.
- $\mathbf{b}^{[l]}$: Bu katmanın **bias vektörü**. Boyut: $(n^{[l]} \times 1)$ — her nörona bir bias.
- $\mathbf{z}^{[l]}$: **Doğrusal kombinasyon** (activation öncesi). Boyut: $(n^{[l]} \times 1)$.
- $\mathbf{a}^{[l]}$: Bu katmanın çıktısı — $\mathbf{z}^{[l]}$'ye aktivasyon uygulanmış hâli.

### Boyut Kontrolü

Matris çarpımında boyutlar çok kritik. Kaç kere "shape mismatch" hatası alacağınızı şimdiden tahmin edebilirim — herkes alır. En garantili çözüm, her adımda boyutları kâğıda yazmaktır.

**Somut örnek:** MNIST sınıflandırıcı, 784-128-64-10 mimarisi.

Tek bir örnek için ileri yayılım (batch size = 1):

| Katman | Notasyon | Boyut | Açıklama |
|---|---|---|---|
| Giriş | $\mathbf{a}^{[0]} = \mathbf{x}$ | $(784, 1)$ | 28×28 görüntü düzleştirilmiş |
| Ağırlık 1 | $\mathbf{W}^{[1]}$ | $(128, 784)$ | İlk gizli katmanın ağırlıkları |
| Bias 1 | $\mathbf{b}^{[1]}$ | $(128, 1)$ | İlk gizli katmanın bias'ları |
| Çıktı 1 | $\mathbf{a}^{[1]}$ | $(128, 1)$ | 128 özellik |
| Ağırlık 2 | $\mathbf{W}^{[2]}$ | $(64, 128)$ | İkinci gizli katman |
| Bias 2 | $\mathbf{b}^{[2]}$ | $(64, 1)$ | |
| Çıktı 2 | $\mathbf{a}^{[2]}$ | $(64, 1)$ | 64 özellik |
| Ağırlık 3 | $\mathbf{W}^{[3]}$ | $(10, 64)$ | Çıkış katmanı |
| Bias 3 | $\mathbf{b}^{[3]}$ | $(10, 1)$ | |
| Çıktı 3 | $\mathbf{a}^{[3]} = \hat{\mathbf{y}}$ | $(10, 1)$ | 10 sınıfa olasılık |

**Toplam parametre sayısı:**
- $W^{[1]}$: $128 \times 784 = 100{,}352$
- $b^{[1]}$: $128$
- $W^{[2]}$: $64 \times 128 = 8{,}192$
- $b^{[2]}$: $64$
- $W^{[3]}$: $10 \times 64 = 640$
- $b^{[3]}$: $10$
- **Toplam: 109,386 parametre**

Yüz bin parametre, ama MNIST için bu küçük bir ağ! GPT-4'ün tahminen **1.76 trilyon** parametresi vardır. Derin öğrenmenin "ölçek" problemi tam da budur.

### Batch İşleme: Tek Seferde Birçok Örnek

Pratikte her örneği tek tek işlemeyiz. Bunun yerine **mini-batch** halinde, diyelim 32 veya 64 örneği aynı anda işleriz.

Artık $\mathbf{X}$'i bir matris olarak düşünüyoruz:
- $\mathbf{X}$ boyutu: $(n^{[0]}, m)$ — burada $m$ batch büyüklüğü

O zaman tek bir ileri yayılım adımı:

$$\mathbf{Z}^{[l]} = \mathbf{W}^{[l]} \mathbf{A}^{[l-1]} + \mathbf{b}^{[l]}$$

(Bias terimi **broadcasting** ile her sütuna eklenir.)

> **⚡ Verimlilik:** NumPy'da matris çarpımı, arka planda optimize edilmiş BLAS kütüphaneleri (OpenBLAS, Intel MKL) kullanılarak yapılır. Bu, saf Python `for` döngüsünden **100-1000 kat** daha hızlıdır. GPU'larda (CUDA, cuBLAS) bu oran binlerce kata çıkar. İşte bu yüzden derin öğrenme GPU'yla patladı.

Şimdi bu matris formülasyonunu kodlayalım:


In [ ]:
# Matris formunda ileri yayilim yapan bir MLP yazalim
# Bu sefer "gercek" sinir agi gibi: bir class, esnek mimari

class SimpleMLP:
    """
    Matris formulasyonu ile cok katmanli perceptron.
    
    Parametreler:
    -----------
    layer_sizes : list of int
        Her katmanin noron sayisi. Ornek: [784, 128, 64, 10]
        ilk eleman giris boyutu, son eleman cikis boyutu.
    activation : str
        'sigmoid', 'tanh', 'relu' den biri.
    """
    
    def __init__(self, layer_sizes, activation='relu'):
        self.layer_sizes = layer_sizes
        self.L = len(layer_sizes) - 1  # katman sayisi (cikis dahil, giris haric)
        
        # Agirliklari ve bias'lari hazirla
        self.W = []  # W[l-1] -> l'inci katmanin agirligi
        self.b = []
        
        np.random.seed(42)
        for l in range(1, len(layer_sizes)):
            n_in = layer_sizes[l-1]
            n_out = layer_sizes[l]
            # Kucuk rastgele agirliklar (He initialization: 2/n_in ile scale)
            W_l = np.random.randn(n_out, n_in) * np.sqrt(2.0 / n_in)
            b_l = np.zeros((n_out, 1))
            self.W.append(W_l)
            self.b.append(b_l)
        
        # Aktivasyon fonksiyonunu ayarla
        if activation == 'sigmoid':
            self.f = lambda x: 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        elif activation == 'tanh':
            self.f = np.tanh
        elif activation == 'relu':
            self.f = lambda x: np.maximum(0, x)
        else:
            raise ValueError(f"Bilinmeyen aktivasyon: {activation}")
    
    def forward(self, X):
        """
        Ileri yayilim.
        
        X : ndarray, shape (n_features, batch_size)
        doner: son katman cikti, shape (n_output, batch_size)
        """
        A = X  # a^[0] = X
        self.cache = [A]  # Geri yayilim icin saklarsak iyi olur (sonraki haftada kullanacagiz)
        
        for l in range(self.L):
            Z = self.W[l] @ A + self.b[l]  # matris carpimi + broadcasting
            A = self.f(Z)
            self.cache.append(A)
        
        return A
    
    def summary(self):
        """Ag mimarisinin ozetini yazdirir."""
        print("=" * 55)
        print(f"{'Katman':<15} {'Boyut':<20} {'Parametre Sayisi':<15}")
        print("=" * 55)
        total = 0
        for l in range(self.L):
            w_params = self.W[l].size
            b_params = self.b[l].size
            layer_total = w_params + b_params
            total += layer_total
            print(f"Katman {l+1:<8} W: {str(self.W[l].shape):<20} {layer_total:,}")
            print(f"{'':15} b: {str(self.b[l].shape):<20}")
        print("=" * 55)
        print(f"Toplam parametre: {total:,}")


# Ornekle: 784-128-64-10 mimarisi
print("MNIST icin bir MLP olusturuyoruz:")
mlp = SimpleMLP([784, 128, 64, 10], activation='relu')
mlp.summary()

# Rastgele bir giris verisi ile ileri yayilim test et
print("\n--- Ileri Yayilim Testi ---")
batch_size = 32
X_test = np.random.randn(784, batch_size)  # 32 goruntu
output = mlp.forward(X_test)
print(f"Giris  sekli:  {X_test.shape}  (784 ozellik, 32 ornek)")
print(f"Cikis  sekli:  {output.shape}  (10 sinif, 32 ornek)")
print(f"Cikisin degerlerinden ornek (ilk ornek, ilk 5 sinif):")
print(f"  {output[:5, 0]}")
print("\n(ReLU cikislari negatif olamayacagi icin hepsi >= 0)")


## 2.7 Kayıp Fonksiyonu Kavramına Kısa Bir Bakış

Dersimizin sonuna yaklaştık ama bir meselemiz var: Yukarıdaki MLP ileri yayılım yapıyor ama **öğrenmiyor**. Ağırlıklar rastgele kaldı. Tahminler de doğal olarak anlamsız.

Öğrenme için önce "ne kadar yanılıyorum?" sorusuna cevap verebilmeliyiz. Bu cevap, **kayıp fonksiyonu** (loss function) ile verilir.

Örneğin basit bir kayıp: Ortalama Kare Hata (Mean Squared Error, MSE):

$$L = \frac{1}{m} \sum_{i=1}^{m} (y_i - \hat{y}_i)^2$$

Burada $y_i$ gerçek etiket, $\hat{y}_i$ ağın tahmini, $m$ örnek sayısı. Her yanılma kare olarak cezalandırılır (pozitiflenir ve büyük hatalar daha ağır cezalandırılır), sonra ortalaması alınır.

Ama asıl soru şu: Kayıp fonksiyonunu hesapladık, peki **ağırlıkları nasıl değiştireceğiz** ki kayıp azalsın? İşte bu, bir sonraki dersimizin konusu: **Gradient Descent** ve **kayıp fonksiyonlarının ayrıntıları**.

---

### 📌 Ders 2 Özeti

| Kavram | Özet |
|---|---|
| **MLP** | Birden fazla perceptron katmanı, her biri öncekinin çıktısını girdi alır |
| **Gizli katman** | Giriş ve çıkış arasındaki "iç" katmanlar; otomatik özellik çıkarır |
| **Universal Approximation** | Yeterli büyüklükte MLP her sürekli fonksiyonu yaklaştırabilir |
| **Aktivasyon** | Doğrusal olmayan fonksiyon; olmasa ağın derinliği anlamsızlaşır |
| **Sigmoid/Tanh** | Klasik ama vanishing gradient sorunu var |
| **ReLU** | Modern ağların standart aktivasyonu; basit ve etkili |
| **GELU** | Transformer'ların tercih ettiği yumuşak ReLU |
| **Matris formu** | $\mathbf{Z}^{[l]} = \mathbf{W}^{[l]} \mathbf{A}^{[l-1]} + \mathbf{b}^{[l]}$ |
| **Batch işleme** | Birden fazla örneği tek matris işleminde hesaplamak — GPU'nun gücü burada |

### 📝 Ödev (Ders 2 için)

1. **Manuel ağırlıklarla XOR**: Yukarıdaki `xor_mlp` fonksiyonunu genişletin ve şu kapıyı manuel ağırlıklarla çözün: **XNOR** (XOR'un tersi).
2. **Aktivasyonlar**: `SimpleMLP` sınıfını her üç aktivasyonla (sigmoid, tanh, ReLU) deneyin. Rastgele bir girdi verdiğinizde çıktılar nasıl farklılaşıyor? Bir inceleme yazın.
3. **Boyut kontrolü**: Şu mimarideki ağın parametre sayısını **elle** hesaplayın, sonra kodla doğrulayın:  
   `[10, 50, 30, 20, 5]`
4. **🔬 İleri düzey**: `SimpleMLP`'ye bir `predict_proba` metodu ekleyin — çıkışa softmax uygulayıp olasılık dağılımı dönsün. (Softmax'ı henüz görmediysek: $\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$)
5. **🤔 Düşünme sorusu**: 1 milyon parametreli bir ağ ile 100 bin parametreli bir ağ, aynı veri üzerinde eğitildiğinde hangisi daha iyi performans gösterir? Her zaman mı? Tartışın.

---


# 📙 Ders 3: Kayıp Fonksiyonları, Gradient Descent ve MNIST Laboratuvarı

*Süre: 45 dakika*

## 3.1 Kayıp Fonksiyonları: Ağın Yanılgısını Ölçmek

Bir sinir ağı eğitmek, aslında bir optimizasyon problemidir:

> **Ağın ağırlıklarını öyle ayarla ki, eğitim verisindeki ortalama hata mümkün olduğunca küçük olsun.**

Ama önce "hata" nedir, onu tanımlamamız gerekiyor. İşte burada **kayıp fonksiyonları** (loss functions) devreye girer.

### 🎯 Ortalama Kare Hata (MSE) — Regresyon için

Regresyon problemlerinde (sayısal bir değer tahmini) en sık kullanılan kayıp:

$$L_{\text{MSE}} = \frac{1}{m} \sum_{i=1}^{m} (y_i - \hat{y}_i)^2$$

**Formülün açıklaması:**
- $m$: Örnek sayısı
- $y_i$: i-inci örneğin **gerçek** değeri
- $\hat{y}_i$: i-inci örneğin **tahmin** edilen değeri
- $(y_i - \hat{y}_i)$: Hata (tahminin gerçekten ne kadar uzakta olduğu)
- Karesini alıyoruz çünkü:
  1. Negatif ve pozitif hatalar birbirini götürmesin
  2. Büyük hatalar orantısız biçimde fazla cezalandırılsın (10'luk hata, 1'lik hatanın 100 katı ceza alır)

> **🎣 Benzetme — Balıkçılık:** Bir olta atıyorsunuz ve balığın konumunu tahmin ediyorsunuz. MSE, "balığı kaçırdığınız mesafenin karesi" gibidir. 1 metre kaçırırsanız 1 puan ceza, 5 metre kaçırırsanız 25 puan. Uzak ıskaları çok cezalandırdığı için, modeliniz "en azından *çok* uzak olmamayı" öğrenir.

### 🎲 Cross-Entropy — Sınıflandırma için

Sınıflandırma problemlerinde (hangi sınıftan olduğu tahmini) MSE kullanmak kötü bir fikirdir. Bunun yerine **cross-entropy** (çapraz entropi) kullanılır.

**Binary (iki sınıflı) cross-entropy:**

$$L_{\text{BCE}} = -\frac{1}{m} \sum_{i=1}^{m} \Big[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \Big]$$

**Formülün açıklaması:**
- $y_i \in \{0, 1\}$: Gerçek etiket
- $\hat{y}_i \in (0, 1)$: Modelin "sınıf 1" için tahmin ettiği olasılık
- Eğer $y_i = 1$ ise sadece ilk terim kalır: $-\log(\hat{y}_i)$. Model 1 demiş ve doğruysa $\hat{y}_i \to 1$, $-\log(1) = 0$ (ceza yok). Model 0 demiş ve yanılmışsa $\hat{y}_i \to 0$, $-\log(0) \to \infty$ (büyük ceza).
- Eğer $y_i = 0$ ise sadece ikinci terim kalır: $-\log(1 - \hat{y}_i)$. Simetrik mantık.

Yani cross-entropy "modelin emin olduğu yanlış tahminleri" ağır cezalandırır.

**Multiclass (çok sınıflı) cross-entropy:**

$$L_{\text{CE}} = -\frac{1}{m} \sum_{i=1}^{m} \sum_{k=1}^{K} y_{i,k} \log(\hat{y}_{i,k})$$

Burada $K$ sınıf sayısı, $y_{i,k}$ **one-hot encoding** ile 1 veya 0 (i-inci örnek k sınıfından mı?), $\hat{y}_{i,k}$ ise modelin k sınıfına verdiği olasılık.

### Softmax: Olasılık Dağılımı Üretmek

Sinir ağının son katmanı ham "logit" değerleri üretir (herhangi bir gerçek sayı). Ama biz olasılık istiyoruz (0 ile 1 arası, toplamı 1). İşte **softmax** bu dönüşümü yapar:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

**Açıklama:** Her logit üstel fonksiyondan geçirilir (böylece pozitif olur ve büyük değerler çok daha büyük olur), sonra tüm üsselleştirilmiş değerlerin toplamına bölünür. Sonuç: tüm çıktılar 0-1 arası ve toplamı tam 1.

> **🎤 Benzetme — Oy Dağıtma:** Diyelim 10 kişilik bir jüri 10 sınıfa puanlar verdi: [3.2, -1.5, 8.1, ..., 2.0]. Bazıları pozitif, bazıları negatif. Softmax bunları "olasılık dilleri" yapar: "sınıf 3'e %73, sınıf 1'e %12, sınıf 5'e %8..." gibi. En yüksek puanı alan sınıf en büyük olasılığı alır (argmax ile seçilir), ama diğer sınıflara da bir olasılık atanır.

### Neden MSE Sınıflandırmada Kullanılmaz?

Yaygın bir soru: Sınıflandırmada da MSE kullanabilirdik, neden kullanmıyoruz? Kısaca:
1. **Olasılıklı yorumu yok** — MSE sınıflandırma için "doğal" bir kayıp değildir.
2. **Vanishing gradient** — MSE + sigmoid kombinasyonunda, çıktı uçlara yakın olduğunda gradyanlar çok küçülür, öğrenme durur.
3. **Cross-entropy** ile ağ yanlış ama emin tahminlerde çok daha büyük gradyan üretir, dolayısıyla hızlı düzelir.


In [ ]:
# Kayip fonksiyonlarini gorsellestirelim
# Binary cross-entropy: gercek etiket y=1 icin, modelin tahmini y_hat degisirken kayip nasil degisir?

y_hat = np.linspace(0.001, 0.999, 500)  # 0 ve 1'den kacinalim (log(0) sorunu)

# y=1 iken: -log(y_hat)
loss_y1 = -np.log(y_hat)

# y=0 iken: -log(1 - y_hat)
loss_y0 = -np.log(1 - y_hat)

# MSE icin (y=1 iken): (1 - y_hat)^2
mse_y1 = (1 - y_hat)**2

# MSE icin (y=0 iken): (y_hat - 0)^2 = y_hat^2
mse_y0 = y_hat**2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: y=1 iken
ax = axes[0]
ax.plot(y_hat, loss_y1, linewidth=2.5, color='#E63946', label='Cross-Entropy: $-\\log(\\hat{y})$')
ax.plot(y_hat, mse_y1, linewidth=2.5, color='#3A86FF', label='MSE: $(1-\\hat{y})^2$', linestyle='--')
ax.set_xlabel('Tahmin edilen olasilik $\\hat{y}$', fontsize=12)
ax.set_ylabel('Kayip', fontsize=12)
ax.set_title('Gercek Etiket $y = 1$ Iken', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 5)
ax.axvline(1, color='green', linestyle=':', alpha=0.5, label='Ideal tahmin')

# Sag: y=0 iken
ax = axes[1]
ax.plot(y_hat, loss_y0, linewidth=2.5, color='#E63946', label='Cross-Entropy: $-\\log(1-\\hat{y})$')
ax.plot(y_hat, mse_y0, linewidth=2.5, color='#3A86FF', label='MSE: $\\hat{y}^2$', linestyle='--')
ax.set_xlabel('Tahmin edilen olasilik $\\hat{y}$', fontsize=12)
ax.set_ylabel('Kayip', fontsize=12)
ax.set_title('Gercek Etiket $y = 0$ Iken', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 5)

plt.suptitle('Cross-Entropy vs MSE: Yanlislari Nasil Cezalandirirlar?',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Gozlemler:")
print("  y=1 iken, y_hat=0.01 (cok yanlis, cok emin):")
print(f"    Cross-entropy kayip: {-np.log(0.01):.2f}  <-- BUYUK ceza!")
print(f"    MSE kayip:            {(1-0.01)**2:.2f}  <-- daha yumusak")
print("\n  y=1 iken, y_hat=0.5 (kararsiz):")
print(f"    Cross-entropy kayip: {-np.log(0.5):.2f}")
print(f"    MSE kayip:            {(1-0.5)**2:.2f}")
print("\nCross-entropy, 'yanlis ve emin' tahminleri cok daha agir cezalandirir.")
print("Bu nedenle siniflandirmada tercih edilir.")


## 3.2 Gradient Descent: Dağdan Aşağı İnmek

Kayıp fonksiyonumuzu tanımladık. Ama bir tek kayıp skoru bizi öğrenmeye götürmez. İhtiyacımız olan şey: **"Kaybı azaltmak için ağırlıkları hangi yönde, ne kadar değiştirmeliyim?"** sorusunun cevabı.

Bu sorunun cevabı **kalkülüsün** en güzel armağanıdır: **gradyan**.

### Gradyanın Sezgisel Anlamı

Diyelim bir dağda sisli bir havada mahsur kaldınız. Aşağı inmek istiyorsunuz ama manzarayı göremiyorsunuz. Ne yaparsınız?

> **Ayaklarınızın altındaki eğimi hissedersiniz**, en dik iniş yönünde küçük bir adım atarsınız, sonra tekrar eğimi hissedip yeni yönde bir adım daha atarsınız.

İşte gradyan descent tam olarak budur. Matematiksel olarak:

$$\theta \leftarrow \theta - \eta \nabla_\theta L$$

**Formülün açıklaması:**
- $\theta$: Tüm parametreler (ağırlıklar ve bias'lar)
- $L$: Kayıp fonksiyonu (tüm verideki ortalama hata)
- $\nabla_\theta L$: Kaybın parametrelere göre **gradyanı** (her parametreye göre kısmi türevlerin vektörü). Bu vektör, kaybın **en hızlı arttığı** yönü gösterir.
- $-\nabla_\theta L$: Ters yön, yani kaybın **en hızlı azaldığı** yön.
- $\eta$: Öğrenme oranı (adım büyüklüğü).

Her iterasyonda, parametreleri gradyanın ters yönünde küçük bir adım kadar güncelleriz. Kayıp azalır. Tekrarla. Tekrarla. Taba ulaşırız (ya da yeterince yakına).

### Gradient Descent Aileleri

**Batch Gradient Descent:** Her adımda **tüm** eğitim verisini kullanarak gradyan hesapla.
- ✅ Gradyan doğru (gürültüsüz)
- ❌ Çok yavaş (milyon örnekli veri için imkânsız)

**Stochastic Gradient Descent (SGD):** Her adımda **bir** örnekle gradyan hesapla.
- ✅ Çok hızlı
- ❌ Gradyan çok gürültülü, oradan oraya sekiyor

**Mini-Batch Gradient Descent:** Her adımda **küçük bir batch** (örn. 32, 64, 128 örnek) kullanarak gradyan hesapla.
- ✅ GPU'lar için ideal (paralel hesap)
- ✅ Gürültü dengeli
- ✅ Modern derin öğrenmenin standardı

> **🍽️ Benzetme — Yemek Tadımı:** Bir şefsiniz ve 1000 kişilik yemek hazırlıyorsunuz. Tadını kontrol etmek için:
> - **Batch**: Tüm 1000 tabağın tadına bakıp sonra tuz ayarlarsınız. (Gerçekçi değil.)
> - **SGD**: Rastgele 1 tabağın tadına bakıp hemen tuz ayarlarsınız. (O tabak temsili değilse kararınız yanlış olur.)
> - **Mini-batch**: Rastgele 32 tabağın ortalama tadına bakıp ayarlarsınız. (En pratik yaklaşım — hem hızlı hem güvenilir.)

### Öğrenme Oranının Etkisi

Aşağıdaki görsel öğrenme oranının tehlikelerini gösterir:


In [ ]:
# Gradient descent'in davranisini gorsellestirelim
# Ornek fonksiyon: f(x) = x^2 + 5 (basit parabol)
# Gradyan: f'(x) = 2x

def f(x):
    return x**2 + 5

def df(x):
    return 2 * x

def gradient_descent_demo(start, lr, n_steps=30):
    """Gradient descent'in izledigi yolu kaydet."""
    path = [start]
    x = start
    for _ in range(n_steps):
        x = x - lr * df(x)
        path.append(x)
    return path

# Uc farkli ogrenme orani
x_plot = np.linspace(-6, 6, 200)
y_plot = f(x_plot)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

scenarios = [
    ('Cok kucuk $\\eta = 0.05$', 0.05, '#3A86FF'),
    ('Ideal $\\eta = 0.3$', 0.3, '#06A77D'),
    ('Cok buyuk $\\eta = 1.02$', 1.02, '#E63946'),
]

for ax, (title, lr, color) in zip(axes, scenarios):
    path = gradient_descent_demo(start=5.0, lr=lr, n_steps=20)
    path_y = [f(p) for p in path]
    
    ax.plot(x_plot, y_plot, 'k-', linewidth=1.5, alpha=0.5, label='$f(x) = x^2 + 5$')
    ax.plot(path, path_y, 'o-', color=color, linewidth=2, markersize=7, 
            label=f'Gradient descent yolu')
    ax.scatter([path[0]], [f(path[0])], color='orange', s=200, zorder=5, 
               edgecolor='black', linewidth=2, label='Baslangic')
    ax.scatter([path[-1]], [f(path[-1])], color='lime', s=200, zorder=5, 
               edgecolor='black', linewidth=2, label='Son nokta')
    
    ax.axhline(5, color='gray', linestyle=':', alpha=0.5, label='Minimum deger')
    ax.set_xlim(-7, 7)
    ax.set_ylim(0, 50)
    ax.set_xlabel('x (parametre)', fontsize=11)
    ax.set_ylabel('f(x) (kayip)', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='upper center')
    ax.grid(True, alpha=0.3)

plt.suptitle('Ogrenme Oraninin Etkisi: Adim Buyuklugu Sonucu Degistiriyor',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Gozlemler:")
print("  Sol:  Cok kucuk lr -> minimuma yaklasir ama cok yavas")
print("  Orta: Ideal lr     -> hizli ve istikrarli bicimde minimuma varir")
print("  Sag:  Cok buyuk lr -> minimumu 'atlar', IRAKSAR (diverges)")
print("\nIyi bir ogrenme orani secimi bir sanattir.")
print("Modern pratikte 'learning rate scheduling' ve 'adaptive optimizers' kullanilir.")


## 3.3 Geri Yayılım (Backpropagation) — Gelecek Haftanın Konusu

Gradyanları nasıl hesaplayacağız? Bir ağırlık için gradyanı düşünün:

$$\frac{\partial L}{\partial W^{[l]}_{ij}}$$

Bu, "$W^{[l]}_{ij}$ ağırlığını küçük miktarda değiştirirsem kaybım ne kadar değişir?" sorusunun cevabıdır. MNIST ağımızda bu tür **109,386 farklı kısmi türev** hesaplamamız gerekiyor! Her biri için ayrı hesap yapsak ömrümüz yetmez.

Neyse ki **geri yayılım** (backpropagation) adlı zekice bir algoritma var. Zincir kuralını akıllıca kullanarak, tüm bu türevleri **tek bir geri geçişte** hesaplıyor. Bu algoritma 1986'da Rumelhart-Hinton-Williams tarafından popülerleştirildi ve modern derin öğrenmenin temelidir.

> **🎁 Sabretin:** Geri yayılımın matematiğini bir sonraki haftaya bırakıyoruz — çünkü hem teoride hem pratikte hakkını vermek için tam bir ders gerekiyor. Bu hafta geri yayılımı bir **kara kutu** olarak kullanacağız: ağımıza veri vereceğiz, o da ağırlıkları kendi başına güncelleyecek.

---

## 3.4 🧪 LABORATUVAR: MNIST Rakamlarını Tanıyan Sinir Ağı

Artık zaman geldi. Şimdiye kadar öğrendiğimiz her şeyi birleştirip **gerçek** bir sinir ağı eğiteceğiz. Göreviniz:

> **28×28 piksellik el yazısı rakam görüntülerinden (0-9) rakamı tanıyan bir MLP eğitin.**

Bu, derin öğrenmenin "Merhaba Dünya"sıdır. MNIST (Modified National Institute of Standards and Technology database) 1998'de Yann LeCun tarafından derlendi ve bugün hâlâ akademik makalelerde baseline olarak kullanılır. İçinde 60,000 eğitim, 10,000 test görüntüsü vardır.

### Plan

1. MNIST'i yükle ve incele
2. Görüntüleri 784'lük vektörlere düzleştir, 0-1 aralığına ölçekle
3. Etiketleri one-hot kodla
4. 784-128-64-10 mimarisinde bir MLP kur
5. Cross-entropy kaybı ile eğit (geri yayılım kodu hazır verilecek)
6. Test doğruluğunu ölç
7. Bazı yanlış sınıflandırmaları görselleştir

### ⚙️ Gerekli Paketler

Eğer yoksa terminal'e: `pip install scikit-learn` yazarak kurun.


In [ ]:
# MNIST veri setini yukleyelim ve inceleyelim
#
# MNIST'i yuklemek icin birkac yontem var - kendi ortaminizda hangisi calisiyorsa onu kullanin:
#   1. tensorflow.keras.datasets.mnist  (en hizli, ayrica offline kurulu geliyor)
#   2. sklearn.datasets.fetch_openml    (openml.org'dan indirir - internet sarttır)
#   3. torchvision.datasets.MNIST       (PyTorch ekosisteminde)
#
# Asagida ilk calisan yontem otomatik secilir.

X_all, y_all = None, None

# Yontem 1: tensorflow/keras
try:
    from tensorflow.keras.datasets import mnist as keras_mnist
    print("MNIST yukleniyor (tensorflow.keras)...")
    (X_train_k, y_train_k), (X_test_k, y_test_k) = keras_mnist.load_data()
    # 28x28 -> 784 duzlestir
    X_all = np.vstack([X_train_k.reshape(-1, 784), X_test_k.reshape(-1, 784)])
    y_all = np.concatenate([y_train_k, y_test_k]).astype(int)
except Exception as e:
    print(f"  tensorflow ile basarisiz: {e}")

# Yontem 2: sklearn
if X_all is None:
    try:
        from sklearn.datasets import fetch_openml
        print("MNIST yukleniyor (sklearn / openml)... (ilk seferde biraz surer)")
        mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
        X_all, y_all = mnist.data, mnist.target.astype(int)
    except Exception as e:
        print(f"  sklearn ile basarisiz: {e}")

if X_all is None:
    raise RuntimeError(
        "MNIST yuklenemedi. Lutfen su paketlerden birini kurun:\n"
        "  pip install tensorflow   (tavsiye edilen)\n"
        "  veya internet baglantinizi kontrol edin."
    )

print(f"\nToplam veri sekli: {X_all.shape}  (70000 goruntu, 784 piksel)")
print(f"Etiket sekli:      {y_all.shape}")
print(f"Piksel degerleri:  {X_all.min():.0f} ile {X_all.max():.0f} arasi (0=siyah, 255=beyaz)")
print(f"Sinif sayisi:      {len(np.unique(y_all))}  (rakamlar 0-9)")

# Her rakamdan kac tane var?
print("\nHer rakamdan kac ornek var:")
unique, counts = np.unique(y_all, return_counts=True)
for digit, count in zip(unique, counts):
    bar = '#' * (count // 200)
    print(f"  {digit}: {count:5d}  {bar}")

# Birkac ornegi gorsellestirelim
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    # Her siniftan bir ornek
    idx = np.where(y_all == i)[0][0]
    img = X_all[idx].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Etiket: {y_all[idx]}', fontsize=12, fontweight='bold')
    ax.axis('off')
plt.suptitle('MNIST: Her Rakamdan Bir Ornek', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Veriyi egitim ve test olarak bolelim, on-isleme yapalim

# Egitim / test bolmesi (ilk 60000 egitim, son 10000 test - MNIST standardi)
X_train_raw = X_all[:60000].astype(np.float32)
y_train = y_all[:60000]
X_test_raw  = X_all[60000:].astype(np.float32)
y_test  = y_all[60000:]

# Normalizasyon: 0-255 -> 0-1
X_train = X_train_raw / 255.0
X_test  = X_test_raw / 255.0

# One-hot encoding (etiketleri vektore cevirelim)
# Ornek: etiket 3 -> [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
def one_hot(y, num_classes=10):
    n = len(y)
    Y = np.zeros((num_classes, n))
    Y[y, np.arange(n)] = 1
    return Y

Y_train = one_hot(y_train)
Y_test  = one_hot(y_test)

# Giris verisini transpoze edelim: (n_samples, 784) -> (784, n_samples)
# Matris formulasyonuna uygun
X_train = X_train.T
X_test  = X_test.T

print("On-isleme sonrasi:")
print(f"  X_train sekli: {X_train.shape}  (784 piksel, 60000 ornek)")
print(f"  Y_train sekli: {Y_train.shape}  (10 sinif, 60000 ornek)")
print(f"  X_test  sekli: {X_test.shape}  (784 piksel, 10000 ornek)")
print(f"  Y_test  sekli: {Y_test.shape}  (10 sinif, 10000 ornek)")

print(f"\nPiksel degerleri artik: {X_train.min():.2f} ile {X_train.max():.2f} arasinda")
print(f"\nOrnek one-hot encoding:")
print(f"  Etiket 3 -> {one_hot(np.array([3])).flatten()}")
print(f"  Etiket 7 -> {one_hot(np.array([7])).flatten()}")


In [ ]:
# Simdi MNIST icin tam-kan bir MLP yazalim
# Geri yayilim burada KARA KUTU - matematigini gelecek hafta goreceğiz

class NeuralNetwork:
    """
    Cok katmanli sinir agi (MLP).
    
    Ileri yayilim: elle yazildi, her detayi anlasilir.
    Geri yayilim: kara kutu (gelecek haftanin konusu).
    Ogrenme:      mini-batch SGD + softmax + cross-entropy
    """
    
    def __init__(self, layer_sizes, learning_rate=0.1):
        self.layer_sizes = layer_sizes
        self.L = len(layer_sizes) - 1
        self.lr = learning_rate
        
        # He initialization (ReLU icin ideal)
        np.random.seed(42)
        self.W = []
        self.b = []
        for l in range(1, len(layer_sizes)):
            n_in = layer_sizes[l-1]
            n_out = layer_sizes[l]
            self.W.append(np.random.randn(n_out, n_in) * np.sqrt(2.0 / n_in))
            self.b.append(np.zeros((n_out, 1)))
    
    def relu(self, Z):
        return np.maximum(0, Z)
    
    def relu_derivative(self, Z):
        return (Z > 0).astype(float)
    
    def softmax(self, Z):
        # Sayisal kararlilik icin max cikarma (overflow onleme)
        Z_shift = Z - np.max(Z, axis=0, keepdims=True)
        exp_Z = np.exp(Z_shift)
        return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)
    
    def forward(self, X):
        """Ileri yayilim - tum katmanlar."""
        self.Z = []   # activation oncesi
        self.A = [X]  # activation sonrasi (a^[0] = X)
        
        A = X
        for l in range(self.L):
            Z_l = self.W[l] @ A + self.b[l]
            self.Z.append(Z_l)
            
            # Son katmanda softmax, oncekilerde ReLU
            if l == self.L - 1:
                A = self.softmax(Z_l)
            else:
                A = self.relu(Z_l)
            self.A.append(A)
        
        return A
    
    def compute_loss(self, Y_pred, Y_true):
        """Cross-entropy kayip."""
        m = Y_true.shape[1]
        # Log(0) onleme icin kucuk epsilon
        eps = 1e-12
        loss = -np.sum(Y_true * np.log(Y_pred + eps)) / m
        return loss
    
    def backward(self, Y_true):
        """
        Geri yayilim. 
        
        DIKKAT: Bu KARA KUTU. Gelecek hafta matemaigine gireriz.
        Simdilik 'bu fonksiyon agirliklari dogru guncelliyor' kabul edelim.
        """
        m = Y_true.shape[1]
        
        # Gradyanlari tutacak listeler
        dW = [None] * self.L
        db = [None] * self.L
        
        # Cikis katmani icin ozel durum: softmax + cross-entropy turevi cok temiz
        dZ = self.A[-1] - Y_true  # (10, batch_size)
        
        for l in reversed(range(self.L)):
            dW[l] = (dZ @ self.A[l].T) / m
            db[l] = np.sum(dZ, axis=1, keepdims=True) / m
            
            if l > 0:
                dA_prev = self.W[l].T @ dZ
                dZ = dA_prev * self.relu_derivative(self.Z[l-1])
        
        return dW, db
    
    def update_params(self, dW, db):
        """Gradyan descent ile agirliklari guncelle."""
        for l in range(self.L):
            self.W[l] -= self.lr * dW[l]
            self.b[l] -= self.lr * db[l]
    
    def predict(self, X):
        """Test icin tahmin - en olasi sinifi dondurur."""
        probs = self.forward(X)
        return np.argmax(probs, axis=0)
    
    def accuracy(self, X, y_true):
        y_pred = self.predict(X)
        return np.mean(y_pred == y_true)


# Modeli olusturalim
model = NeuralNetwork(layer_sizes=[784, 128, 64, 10], learning_rate=0.1)

print("Ag mimarisi olusturuldu:")
print(f"  Katmanlar: {model.layer_sizes}")
print(f"  Toplam katman sayisi (giris haric): {model.L}")
print(f"  Ogrenme orani (eta): {model.lr}")

# Parametre sayisi
total_params = sum(W.size + b.size for W, b in zip(model.W, model.b))
print(f"  Toplam parametre: {total_params:,}")

# Egitim oncesi rastgele test
print("\nEgitim ONCESI test dogrulugu:")
train_acc_before = model.accuracy(X_train[:, :1000], y_train[:1000])
test_acc_before = model.accuracy(X_test, y_test)
print(f"  Egitim  (ilk 1000): {train_acc_before*100:.2f}%")
print(f"  Test   (tum 10000): {test_acc_before*100:.2f}%")
print("  (Rastgele tahmin ~%10 yapardi. Simdi yaklasik o civarindayiz.)")


In [ ]:
# Egitim donusu: mini-batch SGD ile egitelim
import time

def egit(model, X_train, Y_train, y_train, X_test, y_test, 
         epochs=10, batch_size=64):
    """
    Modeli mini-batch SGD ile egitir.
    
    Her epoch: tum egitim verisi uzerinden rastgele sirayla gec,
    her batch icin: forward -> loss -> backward -> update.
    """
    n_samples = X_train.shape[1]
    n_batches = n_samples // batch_size
    
    history = {'loss': [], 'train_acc': [], 'test_acc': []}
    
    print("=" * 60)
    print(f"Egitim basliyor: {epochs} epoch, batch size {batch_size}")
    print(f"Her epoch'ta {n_batches} batch islenecek")
    print("=" * 60)
    
    for epoch in range(epochs):
        t0 = time.time()
        
        # Her epoch basinda veriyi karistir
        perm = np.random.permutation(n_samples)
        X_shuffled = X_train[:, perm]
        Y_shuffled = Y_train[:, perm]
        
        epoch_loss = 0.0
        for b in range(n_batches):
            # Mini-batch al
            start = b * batch_size
            end = start + batch_size
            X_batch = X_shuffled[:, start:end]
            Y_batch = Y_shuffled[:, start:end]
            
            # Ileri yayilim
            Y_pred = model.forward(X_batch)
            
            # Kayip hesapla
            loss = model.compute_loss(Y_pred, Y_batch)
            epoch_loss += loss
            
            # Geri yayilim (gradyanlari bul)
            dW, db = model.backward(Y_batch)
            
            # Agirliklari guncelle
            model.update_params(dW, db)
        
        avg_loss = epoch_loss / n_batches
        
        # Her epoch sonunda dogruluk kontrolu
        train_acc = model.accuracy(X_train[:, :5000], y_train[:5000])
        test_acc = model.accuracy(X_test, y_test)
        
        history['loss'].append(avg_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        
        elapsed = time.time() - t0
        print(f"Epoch {epoch+1:2d}/{epochs} | "
              f"Kayip: {avg_loss:.4f} | "
              f"Train acc: {train_acc*100:.2f}% | "
              f"Test acc: {test_acc*100:.2f}% | "
              f"Sure: {elapsed:.1f}s")
    
    print("=" * 60)
    return history


# Egitelim!
history = egit(model, X_train, Y_train, y_train, X_test, y_test,
               epochs=10, batch_size=64)

print(f"\nSon test dogrulugu: {history['test_acc'][-1]*100:.2f}%")
print("(Iyi bir baseline: ~%97 civari MNIST icin basit bir MLP ile elde edilebilir.)")


In [ ]:
# Egitim surecini gorsellestirelim

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['loss']) + 1)

# Sol: Kayip egrisi
ax = axes[0]
ax.plot(epochs_range, history['loss'], 'o-', linewidth=2.5, markersize=8,
        color='#E63946', label='Egitim Kaybi')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Cross-Entropy Kayip', fontsize=12)
ax.set_title('Kayip Azaliyor -> Model Ogreniyor!', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Sag: Dogruluk egrisi
ax = axes[1]
ax.plot(epochs_range, [a*100 for a in history['train_acc']], 'o-', 
        linewidth=2.5, markersize=8, color='#3A86FF', label='Egitim Dogrulugu')
ax.plot(epochs_range, [a*100 for a in history['test_acc']], 's-', 
        linewidth=2.5, markersize=8, color='#06A77D', label='Test Dogrulugu')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Dogruluk (%)', fontsize=12)
ax.set_title('Dogruluk Artiyor -> Tahminler Iyilesiyor!', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim(80, 100)

plt.suptitle('MNIST Uzerinde Egitim Sureci', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Gozlem:")
print("  - Kayip monotonik olarak azaliyor (guzel)")
print("  - Egitim ve test dogruluklari birbirine yakin")
print("  - Eger egitim >> test olsaydi 'overfitting' olurdu (dersten ezberleme)")
print("  - Eger ikisi de dusukse 'underfitting' olurdu (yeterince ogrenememis)")


## 3.5 Modelin Davranışını İnceleme

Artık %97 doğruluk elde ettiğimize göre, "model nerede hatalı?" diye bakmanın zamanı. Bu, derin öğrenmedeki en önemli alışkanlıklardan biridir — **her zaman** modelinizin nerede yanıldığına bakın. Genellikle bu hatalar bir şey öğretir.

### Confusion Matrix (Karışıklık Matrisi)

Bu matris, "hangi rakamlar hangisiyle karıştırılıyor" sorusunun cevabını verir. 10×10 bir tablo; satırlar gerçek etiketler, sütunlar tahminler. Köşegen üzerindeki değerler doğru tahminlerdir, dışındakiler hatalar.


In [ ]:
# Confusion matrix hesaplayalim
y_pred_test = model.predict(X_test)

# Confusion matrix'i NumPy ile elle hesaplayalim
cm = np.zeros((10, 10), dtype=int)
for true, pred in zip(y_test, y_pred_test):
    cm[true, pred] += 1

# Gorsellestir
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues', aspect='auto')

# Hucrelere sayilari yaz
for i in range(10):
    for j in range(10):
        val = cm[i, j]
        # Kosegen uzerindekiler koyu mavi olur, metni beyaz yaz
        color = 'white' if val > cm.max() / 2 else 'black'
        weight = 'bold' if i == j else 'normal'
        ax.text(j, i, str(val), ha='center', va='center', 
                color=color, fontsize=10, fontweight=weight)

ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(range(10))
ax.set_yticklabels(range(10))
ax.set_xlabel('Tahmin Edilen Sinif', fontsize=13, fontweight='bold')
ax.set_ylabel('Gercek Sinif', fontsize=13, fontweight='bold')
ax.set_title('Confusion Matrix (10000 Test Ornegi)\nKosegen = Dogru, Digerleri = Hata',
             fontsize=13, fontweight='bold')

plt.colorbar(im, ax=ax, label='Ornek Sayisi')
plt.tight_layout()
plt.show()

# En sik karistirilan rakam ciftlerini bul
print("\nEn sik karistirilan rakam ciftleri:")
errors = []
for i in range(10):
    for j in range(10):
        if i != j and cm[i, j] > 0:
            errors.append((cm[i, j], i, j))
errors.sort(reverse=True)
for count, true, pred in errors[:5]:
    print(f"  Gercek {true}, ama {pred} dedi: {count} kez")

print("\nBu bize ne soyluyor?")
print("  Modelin en cok zorlandigi ciftler genellikle gorsel olarak benzer rakamlar:")
print("  (4,9), (3,5), (7,2) gibi - el yazisinda gercekten zor ayristirilabilirler.")


In [ ]:
# Yanlis siniflandirilan bazi ornekleri inceleyelim
# Bu cok ogretici - bazen modelin yanilgisini anlamak kolay (ilim gercekten belirsiz),
# bazen ise sasirtici hatalar yapar (bu, modeli iyilestirme ipucu verir).

# Yanlis tahmin edilen indisleri bul
wrong_idx = np.where(y_pred_test != y_test)[0]
print(f"Toplam yanlis tahmin: {len(wrong_idx)} / {len(y_test)}")
print(f"Hata orani: {len(wrong_idx)/len(y_test)*100:.2f}%")

# 15 tanesini rastgele sec ve goster
np.random.seed(7)
sample_wrong = np.random.choice(wrong_idx, size=15, replace=False)

fig, axes = plt.subplots(3, 5, figsize=(14, 9))
for i, idx in enumerate(sample_wrong):
    ax = axes[i // 5, i % 5]
    
    # X_test (784, n_samples) seklinde, onun icin transpoze edip reshape et
    img = X_test[:, idx].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    
    # Modelin olasiliklarini da gosterelim
    probs = model.forward(X_test[:, idx:idx+1]).flatten()
    top2_idx = np.argsort(probs)[::-1][:2]
    
    ax.set_title(
        f"Gercek: {y_test[idx]}  |  Tahmin: {y_pred_test[idx]}\n"
        f"1.: {top2_idx[0]} (%{probs[top2_idx[0]]*100:.0f})  "
        f"2.: {top2_idx[1]} (%{probs[top2_idx[1]]*100:.0f})",
        fontsize=9
    )
    ax.axis('off')

plt.suptitle('Modelin Yanildigi 15 Ornek\n(Bazilari insana bile zor gelir!)',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\nDikkat: Bazi yanlis tahminlerde model bile kararsiz (1. ve 2. tahmin yakin).")
print("Bu, modelin 'belirsizligi' yansittigini gosterir - iyi bir ozellik!")


## 3.6 Ne Yaptık? Bir Zafer Anı

Durun ve düşünün. Bu haftanın başlangıcında, bir Sobel filtresinin değerlerini elle seçmenin zor olduğundan şikâyet ediyorduk. Bu hafta sonunda:

- **Sıfırdan**, bir sinir ağı inşa ettik (hiçbir framework kullanmadan, sadece NumPy!)
- Bu ağ, **kendi başına**, 60,000 el yazısı rakamdan hangi pikselin önemli olduğunu **öğrendi**
- Daha önce görmediği 10,000 rakam üzerinde **%97 doğrulukla** tahmin yapabiliyor

Bu, Frank Rosenblatt'ın 1958'de hayal ettiği şeyin tam kendisi. Ve bu sadece başlangıç.

> **🏛️ Tarihsel perspektif:** 1998'de Yann LeCun bu aynı MNIST problemi üzerinde LeNet-5 adlı bir CNN ile **%99.2** doğruluk elde etti. O zamanlar bu akıl almaz bir başarıydı. Bugün, ders bitmeden MNIST'te %97 alabiliyoruz. Bir sonraki haftalarda CNN'leri öğrendiğimizde, biz de %99'un üzerine çıkacağız.

---

## 3.7 Henüz Değinmediğimiz Ama Çok Önemli Konular

Bu hafta **temelleri** kurduk. Ama derin öğrenmede doğru öğrenmek için daha pek çok şey var. İlerideki haftalarda göreceklerimizden bazıları:

### 1. Geri Yayılımın Matematiği (Hafta 9)
Bu hafta `backward()` fonksiyonunu kara kutu olarak kullandık. Önümüzdeki hafta onun içini açıp, zincir kuralıyla nasıl tüm gradyanları verimli bir şekilde hesapladığımızı göreceğiz.

### 2. Optimizerlar: SGD'nin Ötesinde (Hafta 9)
Düz SGD güzel ama yavaş. **Momentum**, **RMSprop**, **Adam** gibi modern optimizerlar çok daha hızlı yakınsar. Adam bugün fiili standarttır.

### 3. Overfitting ve Regülarizasyon (Hafta 9-10)
Ağ çok büyük olunca eğitim verisini "ezberleyebilir" (overfitting). Bunun önüne geçmek için **dropout**, **L2 regularization**, **data augmentation** gibi teknikler vardır.

### 4. Konvolüsyonel Sinir Ağları (Hafta 10)
Bizim kullandığımız MLP, 28×28'lik küçük görüntüler için çalışıyor, ama 224×224 renkli bir görüntü için 150,528 giriş nöronu gerekir — parametreler patlar. **CNN'ler** bu problemi çözmek için icat edildi: aynı filtreyi görüntünün her yerinde uygulayarak, uzamsal bilgiyi koruyarak, çok daha az parametreyle.

### 5. Modern Mimariler (Hafta 10-14)
ResNet (artık bağlantılar), Transformer'lar (attention mekanizması), Stable Diffusion (gürültü azaltma), SAM (segment anything)... Her biri bir önceki haftanın üzerine kurulur.

---

## 📌 Hafta 8 — Büyük Resim Özeti

| Hafta 7'ye Kadar | Hafta 8'de Öğrendiğimiz |
|---|---|
| Elle tasarlanmış filtreler | Öğrenilmiş ağırlıklar |
| Sobel, Gauss, HOG, SIFT | Perceptron, MLP |
| `f(x) = ?` (biz biliyoruz) | `f(x) = ?` (ağ öğreniyor) |
| Klasik görüntü işleme | Sinir ağlarının temelleri |

Artık derin öğrenme yolculuğunun başındasınız. Önümüzdeki 6 hafta boyunca bu temelin üzerine çok daha karmaşık ve güçlü modeller inşa edeceğiz.

---

## 📝 Final Ödevi (Bu Haftanın Tamamı İçin)

### Zorunlu Bölüm

1. **Mimari deneyleri**: MNIST üzerinde şu ağ mimarilerini deneyin ve test doğruluklarını karşılaştırın:
   - `[784, 10]` (gizli katmansız, sadece lojistik regresyon)
   - `[784, 32, 10]` (küçük tek gizli katman)
   - `[784, 128, 64, 10]` (bizim kullandığımız)
   - `[784, 512, 256, 128, 10]` (daha derin)

2. **Aktivasyon karşılaştırması**: Aynı mimariyi (`[784, 128, 64, 10]`) sigmoid, tanh ve ReLU aktivasyonlarıyla ayrı ayrı eğitin. Hangi daha hızlı yakınsıyor?

3. **Öğrenme oranı taraması**: `lr ∈ {0.001, 0.01, 0.1, 0.5, 1.0, 5.0}` için eğitim yapın. Hangisi en iyi? Hangisi iraksıyor? Rapor yazın.

### Serbest Bölüm (Bonus)

4. **🎨 Kendi ağınızı çizin**: Ağın **gizli katmanındaki** bir nöronun hangi piksellere tepki verdiğini görselleştirin. (İpucu: `W[0]` matrisinin bir satırını 28×28 olarak reshape edin ve imshow ile gösterin.) Bu, "nöronun öğrendiği özellik"tir.

5. **🔬 Kendi veri setinizi oluşturun**: Kendi el yazınızla 0-9 arası rakamlar yazın, fotoğraf çekin, 28×28'e resize edin ve modelinize tahmin ettirin. Ne oluyor? Model sizin el yazınızı tanıyor mu?

6. **📚 Okuma**: Michael Nielsen'in *Neural Networks and Deep Learning* kitabının ilk iki bölümünü okuyun (ücretsiz: neuralnetworksanddeeplearning.com). Okuma sonrası 1 sayfalık düşünce notu yazın.

---

## 🎯 Sonsöz

Bu defter üç dersin akışını yakalamak için tasarlandı. Ama asıl öğrenme tek tek hücreleri çalıştırıp, ağırlıkları değiştirip, "ne olursa?" diye merak ettiğinizde oluşacak. Feynman'ın sözüyle başladık; aynı sözle bitirelim:

> *"What I cannot create, I do not understand."*

Bir sinir ağını kendi ellerinizle yazdınız. Artık onu anladığınızı söyleyebilirsiniz.

**Sonraki haftada görüşmek üzere — geri yayılımın matematiğine dalacağız! 🚀**

---

### 📚 İleri Okumalar

- **Michael Nielsen**, *Neural Networks and Deep Learning* — neuralnetworksanddeeplearning.com (ücretsiz, interaktif)
- **Ian Goodfellow, Yoshua Bengio, Aaron Courville**, *Deep Learning* (2016) — deeplearningbook.org (ücretsiz PDF)
- **3Blue1Brown** YouTube serisi: "Neural Networks" (görsel sezgi için mükemmel)
- **Andrew Ng**, Coursera Deep Learning Specialization (daha yapılandırılmış öğrenme için)
- **Rosenblatt (1958)**, "The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain" — orijinal makale, okumaya değer bir tarihsel parça
- **Minsky & Papert (1969)**, *Perceptrons* — 50 yıl sonra bile okumakta fayda var
- **Rumelhart, Hinton, Williams (1986)**, "Learning representations by back-propagating errors" — derin öğrenmenin dönüm noktası makalesi
